In [100]:
!which python
!python --version

/home/commander-data/Storage/users/mave/.venv/bin/python
Python 3.12.9


In [ ]:
import pandas as pd
import seaborn as sns
import re
import pysam
import functions
import numpy as np
import gffutils
import os

<module 'functions' from '/home/commander-data/Storage/users/mave/search_sequences/library_design/functions.py'>

In [102]:
gene = {
    "symbol": "NF1",
    "ensembl_gene_id": "ENSG00000196712",
    "chrom": "chr17",
    "start": 31094977,
    "end": 31377675,
    "mane_select": "ENST00000358273", # version 9
    "refseq_select": "NM_001042492", # version 3
    "strand": "+"
}

In [ ]:
## set paths
basepath = "/home/commander-data/Storage/users/mave"
genome_data_path = basepath + "/genome_data"
transcripts_path = genome_data_path + "/ensembl_transcripts/115"
ensembl_gff_db = transcripts_path + '/Homo_sapiens.GRCh38.115.db' # create the database using create_gff_db.py script
genome_path = genome_data_path + "/genomes/grch38.fa"

# gene specific gff paths
transcripts_oi_path = genome_data_path + "/genes"
os.makedirs(transcripts_oi_path, exist_ok=True)
transcripts_oi_file = transcripts_oi_path + "/ensembl_" + gene["symbol"] + ".gff3"

# splice prediction paths
work_path = basepath + "/search_sequences"
datapath = work_path + "/data/library_design"
pangolin_scores_path = datapath + "/splicing_prediction_exons_consequence.vcf"
spliceai_scores_path = datapath + "/splicing_prediction_exons_spliceai.vcf"

In [ ]:
# setup data access
genome = pysam.FastaFile(genome_path)
ensembl_gff = gffutils.FeatureDB(ensembl_gff_db, keep_order=True)

In [ ]:
# define some binning functions
def decide_final_tag(gff_entry): # for transcript tags
    if "tag" not in gff_entry.attributes:
        return "other"
    tags = gff_entry.attributes["tag"]
    if "MANE_Select" in tags:
        return "MANE_Select"
    if "gencode_primary" in tags:
        return "gencode_primary"
    if "Ensembl_canonical" in tags:
        return "Ensembl_canonical"
    if "gencode_basic" in tags:
        return "gencode_basic"
    return "other"

def decide_bin(score): # for splice predictions
    if score > 0.7:
        return "high_gain"
    if score > 0.4:
        return "mid_gain"
    if score > 0.2:
        return "low_gain"
    if score < -0.7:
        return "high_loss"
    if score < -0.4:
        return "mid_loss"
    if score < -0.2:
        return "low_loss"
    return "no_splicing"

In [ ]:
# read in exons
exons_oi = None
for exon_id, cds in enumerate(ensembl_gff.children('transcript:' + gene["mane_select"], featuretype="CDS", order_by="start")):
    start = cds.start
    end = cds.end
    chrom = cds.chrom
    new_data = pd.DataFrame([chrom, start, end, exon_id + 1]).T

    if exons_oi is None:
        exons_oi = new_data
    else:
        exons_oi = pd.concat([exons_oi, new_data], axis = 0)
exons_oi.columns = ["chrom", "start", "end", "exon_id"]
exons_oi = exons_oi.set_index("exon_id")

exons_oi["length"] = exons_oi["end"] - exons_oi["start"] + 1
exons_oi["cdna_end"] = exons_oi["length"].cumsum()
exons_oi["cdna_start"] = exons_oi["cdna_end"] - exons_oi["length"]
exons_oi["cdna_start"] = exons_oi["cdna_start"] + 1
exons_oi

,chrom,start,end,length,cdna_end,cdna_start
exon_id,,,,,,
1,17,31095310,31095369,60,60,1
2,17,31155983,31156126,144,204,61
3,17,31159010,31159093,84,288,205
4,17,31163186,31163376,191,479,289
5,17,31169891,31169997,107,586,480
6,17,31181422,31181489,68,654,587
7,17,31181710,31181785,76,730,655
8,17,31182508,31182665,158,888,731
9,17,31200422,31200595,174,1062,889


In [ ]:
# save gene specific gff file
with open(transcripts_oi_path, "w") as transcripts_oi_file:
    for x in ensembl_gff.children('gene:' + gene["ensembl_gene_id"]):
        x.attributes["final_tag"] = decide_final_tag(x)
        transcripts_oi_file.write(str(x) + "\n")
        #if x.attributes["ID"][0] == 'transcript:ENST00000358273':
        #    test = x
        for y in ensembl_gff.children(x.attributes["ID"][0]):
            transcripts_oi_file.write(str(y) + "\n")

CREATE DATA FOR LIBRARY

In [ ]:
# annotate spliceai vcf
outfile = open(datapath + "/splicing_prediction_exons_spliceai_severe.vcf", "w")

with open(spliceai_scores_path, "r") as f:
    for line in f:
        if line.strip() == '':
            continue
        if line.startswith('#'):
            outfile.write(line)
            continue

        line = line.strip()
        parts = line.split('\t')
        info = parts[7]

        spliceai = functions.find_between(info, "SpliceAI=", '(;|$)')
        spliceai_parts = spliceai.split('|') # ALLELE|SYMBOL|DS_AG|DS_AL|DS_DG|DS_DL|DP_AG|DP_AL|DP_DG|DP_DL

        ag = float(spliceai_parts[2])
        al = float(spliceai_parts[3]) * -1
        dg = float(spliceai_parts[4])
        dl = float(spliceai_parts[5]) * -1

        most_severe_gain = max(ag, dg)
        most_severe_loss = min(al, dl)

        most_severe_score = most_severe_gain
        if abs(most_severe_loss) > most_severe_gain:
            most_severe_score = most_severe_loss
        
        if abs(most_severe_score) < 0.2:
            continue

        spliceai_bin = decide_bin(most_severe_score)

        info_parts = info.split(';')
        info_parts.append("spliceai_most_severe_score=" + str(most_severe_score))
        info_parts.append("spliceai_bin=" + spliceai_bin)
        info = ';'.join(info_parts)

        parts[7] = info
        outfile.write('\t'.join(parts) + "\n")

outfile.close()

In [ ]:
# annotate pangolin vcf
outfile = open(datapath + "/splicing_prediction_exons_pangolin_severe.vcf", "w")

with open(pangolin_scores_path, "r") as f:
    for line in f:
        if line.strip() == '':
            continue
        if line.startswith('#'):
            outfile.write(line)
            continue

        line = line.strip()
        parts = line.split('\t')
        info = parts[7]

        pangolin = functions.find_between(info, "Pangolin=", '(;|$)')
        pangolin_parts = pangolin.split('|') # gene|pos:score_change|pos:score_change|warnings,...

        consequences = functions.find_between(info, "CSQ=", "(;|$)") # Allele|Consequence|IMPACT|SYMBOL|HGNC_ID|Feature|Feature_type|EXON|INTRON|HGVSc|HGVSp, komma separated
        consequences = consequences.split(",")
        consequence_oi = ""
        hgvsp_oi = ""
        for consequence in consequences:

            consequence_parts = consequence.split('|')
            consequence_str = consequence_parts[1]
            hgvsp_str = consequence_parts[10]
            transcript = consequence_parts[5].split('.')[0]
            if transcript == gene["mane_select"]:
                consequence_oi = consequence_str
                hgvsp_oi = hgvsp_str


        gain = float(pangolin_parts[1].split(':')[1])
        loss = float(pangolin_parts[2].split(':')[1])

        most_severe_score = gain
        if abs(loss) > gain:
            most_severe_score = loss
        
        if abs(most_severe_score) < 0.2:
            continue

        pangolin_bin = decide_bin(most_severe_score)

        info_parts = info.split(';')
        info_parts.append("pangolin_most_severe_score=" + str(most_severe_score))
        info_parts.append("pangolin_bin=" + pangolin_bin)
        info_parts.append("consequence=" + consequence_oi)
        info_parts.append("hgvsp=" + hgvsp_oi)
        info = ';'.join(info_parts)

        parts[7] = info
        outfile.write('\t'.join(parts) + "\n")

outfile.close()

A|stop_gained|HIGH|NF1|HGNC:7765|ENST00000358273.9|Transcript|42/58||c.6165C>A|p.Cys2055Ter


In [ ]:
## convert pangolin tsv precalculated scores to pangolin vcf for a specific gene
#pangolin_scores_path = datapath + "/Pangolin_hg38_snvs_masked"
#pangolin_scores = pd.read_csv(pangolin_scores_path + "/" + gene["ensembl_gene_id"] + ".tsv.gz", compression = "gzip", sep = "\t")
#pangolin_scores = pangolin_scores[(pangolin_scores["gain_score"] > 0.2) | (pangolin_scores["loss_score"] < -0.2)]
#pangolin_scores = pangolin_scores.reset_index(drop = True)
#pangolin_scores["most_severe_score"] = pangolin_scores.apply(lambda x: x["gain_score"] if x["gain_score"] >= abs(x["loss_score"]) else x["loss_score"], axis = 1)
#pangolin_scores["pangolin_bin"] = [decide_bin(x) for x in pangolin_scores["most_severe_score"]]
#
## save to vcf file
#outfile = open(datapath + "/severe_pangolin_nf1.vcf", "w")
#
#header_lines = [
#    '##fileformat=VCFv4.1',
#    '##fileDate=2025-08-31',
#    '##reference=GRCh38',
#    '##ID=<Description="Severe Pangolin scores">',
#    '##INFO=<ID=gain_score,Number=1,Type=Float,Description="The pangolin gain score">',
#    '##INFO=<ID=gain_pos,Number=1,Type=Integer,Description="The position with the largest pangoliln gain score">',
#    '##INFO=<ID=loss_score,Number=1,Type=Float,Description="The pangolin loss score">',
#    '##INFO=<ID=loss_pos,Number=1,Type=Integer,Description="The position with the largest pangoliln loss score">',
#    '#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO'
#]
#
#for line in header_lines:
#    outfile.write(line + "\n")
#
#for index in pangolin_scores.index:
#    current_data = pangolin_scores.loc[index, :]
#
#    info = [
#        "pangolin_gain_score=" + str(current_data["gain_score"]),
#        "pangolin_gain_pos=" + str(current_data["gain_pos"]),
#        "pangolin_loss_score=" + str(current_data["loss_score"]),
#        "pangolin_loss_pos=" + str(current_data["loss_pos"]),
#        "pangolin_bin=" + str(current_data["pangolin_bin"]),
#        "Pangolin_masked=" + '|'.join([
#            "ENSG00000196712",
#            str(current_data["gain_pos"]) + ":" + str(current_data["gain_score"]),
#            str(current_data["loss_pos"]) + ":" + str(current_data["loss_score"]),
#            ""
#        ])
#    ]
#    info = ';'.join(info)
#
#    new_line = [
#        current_data["chrom"],
#        current_data["pos"],
#        ".",
#        current_data["ref"],
#        current_data["alt"],
#        ".",
#        ".",
#        info
#    ]
#    new_line = [str(x) for x in new_line]
#
#    outfile.write('\t'.join(new_line) + "\n") #CHROM POS      ID         REF   ALT    QUAL  FILTER   INFO
#
#outfile.close()

In [ ]:
## filter nf1 vcf to only coding mutations
#script_path = basepath + "/search_sequences/library_design/filter_vcf.py"
#all_vars_path = basepath + "/search_sequences/data/library_design/splicing_prediction.vcf"
#only_exons_path = basepath + "/search_sequences/data/library_design/splicing_prediction_exons.vcf"
#
#exon_boundaries_oi = [(x.start-10, x.end+10) for x in ensembl_gff.children('transcript:' + gene["mane_select"], featuretype="CDS")]
#
#first_boundaries = exon_boundaries_oi[0]
#start = first_boundaries[0]
#end = first_boundaries[1]
#!python3 $script_path -i $all_vars_path -c chr17 -s $start -e $end -k > $only_exons_path
#
#for start, end in exon_boundaries_oi[1:len(exon_boundaries_oi)]:
#    !python3 $script_path -i $all_vars_path -c chr17 -s $start -e $end >> $only_exons_path

final library design

In [ ]:
#tools/ngs_bits/bin/VcfAnnotateConsequence -in search_sequences/data/library_design/splicing_prediction_exons_pangolin.vcf -out search_sequences/data/library_design/splicing_prediction_exons_consequence.vcf -gff tools/ngs_bits/ngsd_data/Homo_sapiens.GRCh38.110.gff3 -ref genome_data/genomes/grch38.fa

pangolin_vcf_path = datapath + "/splicing_prediction_exons_pangolin_severe.vcf"
spliceai_vcf_path = datapath + "/splicing_prediction_exons_spliceai_severe.vcf"
clinvar_vcf_path = datapath + "/clinvar_nf1_annotated.vcf"

In [104]:
exon_boundaries_oi = [(x.start, x.end) for x in ensembl_gff.children('transcript:' + gene["mane_select"], featuretype="CDS")]
exon_boundaries_oi = exon_boundaries_oi[1:len(exon_boundaries_oi)-1] # remove the first and last exon

In [ ]:
# collect spliceai scores
spliceai_scores = {}
with open(spliceai_scores_path, "r") as f:
    for line in f:
        if line.strip() == '' or line.startswith('#'):
            continue

        line = line.strip()
        parts = line.split('\t')
        chrom = parts[0]
        pos = parts[1]
        ref = parts[3]
        alt = parts[4]
        info = parts[7]

        var_id = '-'.join([chrom, pos, ref, alt])

        spliceai = functions.find_between(info, "SpliceAI=", '(;|$)')

        if spliceai is None:
            continue

        spliceai_parts = spliceai.split('|') # ALLELE|SYMBOL|DS_AG|DS_AL|DS_DG|DS_DL|DP_AG|DP_AL|DP_DG|DP_DL

        ag_score = round(float(spliceai_parts[2]), 2)
        ag_pos = int(spliceai_parts[6])
        al_score = round(float(spliceai_parts[3]) * -1, 2)
        al_pos = int(spliceai_parts[7])
        dg_score = round(float(spliceai_parts[4]), 2)
        dg_pos = int(spliceai_parts[8])
        dl_score = round(float(spliceai_parts[5]) * -1, 2)
        dl_pos = int(spliceai_parts[9])

        most_severe_gain_score = ag_score
        most_severe_gain_pos = ag_pos
        if al_score > ag_score:
            most_severe_gain_score = al_score
            most_severe_gain_pos = al_pos
        
        most_severe_loss_score = dg_score
        most_severe_loss_pos = dg_pos
        if dl_score < dg_score:
            most_severe_loss_score = dl_score
            most_severe_loss_pos = dl_pos

        #if most_severe_gain_score > abs(most_severe_loss_score) and most_severe_gain_score > 0.2:
        spliceai_scores[var_id] = {
            "spliceai_gain": most_severe_gain_score, 
            "spliceai_gain_pos": most_severe_gain_pos,
            "spliceai_loss": most_severe_loss_score, 
            "spliceai_loss_pos": most_severe_loss_pos, 
            "spliceai": spliceai
        }

# collect pangolin scores and consequences
pangolin_scores = {}
with open(pangolin_scores_path, "r") as f:
    for line in f:
        if line.strip() == '' or line.startswith('#'):
            continue

        line = line.strip()
        parts = line.split('\t')
        chrom = parts[0]
        pos = parts[1]
        ref = parts[3]
        alt = parts[4]
        info = parts[7]

        var_id = '-'.join([chrom, pos, ref, alt])

        pangolin = functions.find_between(info, "Pangolin=", '(;|$)')

        if pangolin is None:
            continue

        pangolin_parts = pangolin.split('|')
        pangolin_gain = pangolin_parts[1].split(':')
        pangolin_gain_score = round(float(pangolin_gain[1]), 2)
        pangolin_gain_pos = int(pangolin_gain[0])
        pangolin_loss = pangolin_parts[2].split(':')
        pangolin_loss_score = round(float(pangolin_loss[1]), 2)
        pangolin_loss_pos = int(pangolin_loss[0])

        consequences = functions.find_between(info, "CSQ=", "(;|$)") # Allele|Consequence|IMPACT|SYMBOL|HGNC_ID|Feature|Feature_type|EXON|INTRON|HGVSc|HGVSp, komma separated
        consequences = consequences.split(",")
        consequence_oi = ""
        for consequence in consequences:
            consequence_parts = consequence.split('|')
            consequence_str = consequence_parts[1]
            transcript = consequence_parts[5].split('.')[0]
            if transcript == gene["mane_select"]:
                hgvsc = consequence_parts[9]
                hgvsp = consequence_parts[10]
                consequence_oi = consequence_str
                
        #if pangolin_gain_score > abs(pangolin_loss_score) and pangolin_gain_score > 0.2:
        pangolin_scores[var_id] = {
            "pangolin_gain": pangolin_gain_score,
            "pangolin_gain_pos": pangolin_gain_pos, 
            "pangolin_loss": pangolin_loss_score, 
            "pangolin_loss_pos": pangolin_loss_pos,
            "consequence": consequence_oi,
            "hgvsc": hgvsc,
            "hgvsp": hgvsp
        }

all_splice_positions = set([x.split("-")[1] for x in pangolin_scores.keys()])

# annotate clinvar
clinvar_variants = {}
with open(clinvar_vcf_path, "r") as clinvar_file:
    for line in clinvar_file:
        if line.strip() == "" or line.startswith('#'):
            continue

        parts = line.split('\t')
        ##CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
        chrom = parts[0]
        if not chrom.startswith('chr'):
            chrom = "chr" + chrom
        pos = parts[1]
        ref = parts[3]
        alt = parts[4]
        info = parts[7]

        if len(ref) != 1 or len(alt) != 1:
            continue

        var_id = '-'.join([chrom, pos, ref, alt])
        if pos not in all_splice_positions: # skip all variants which are not at predicted splice positions
            continue
        
        clinvar = functions.find_between(info, "CLNSIG=", '(;|$)')

        clinvar_variants[var_id] = {
            "clinvar": clinvar
        }


In [ ]:
# combine the collected information
all_splice_variants = set(pangolin_scores.keys()) | set(clinvar_variants.keys())
library = []
for splice_variant in all_splice_variants:
    new_line = {"var_id": splice_variant}
    pangolin_score = pangolin_scores.get(splice_variant, {})
    new_line.update(pangolin_score)
    spliceai_score = spliceai_scores.get(splice_variant, {})
    new_line.update(spliceai_score)
    clinvar = clinvar_variants.get(splice_variant, {})
    new_line.update(clinvar)
    
    pos = int(splice_variant.split('-')[1])

    ex_int = "intronic"
    distance_to_ss = 0
    distance_to_ss_left = 0
    distance_to_ss_right = 0
    exon_number = None
    for current_exon_number, exon_boundary in enumerate(exon_boundaries_oi):
        if exon_boundary[0] <= pos <= exon_boundary[1]:
            ex_int = "exonic"
            distance_to_ss_left = pos - exon_boundary[0]
            distance_to_ss_right = exon_boundary[1] - pos
            distance_to_ss = min(distance_to_ss_left, distance_to_ss_right)
            exon_number = current_exon_number + 2
    new_line.update({
        "distance_to_ss": distance_to_ss,
        "distance_to_ss_left": distance_to_ss_left,
        "distance_to_ss_right": distance_to_ss_right,
        "exon_number": exon_number
    })

    library.append(new_line)

library = pd.DataFrame(library)
library

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,spliceai_loss,spliceai_loss_pos,spliceai,distance_to_ss,distance_to_ss_left,distance_to_ss_right,exon_number,clinvar
0,chr17-31252980-A-T,0.00,-12,-0.28,-42,stop_gained,c.4153A>T,p.Lys1385Ter,0.00,20,-0.18,20,T|NF1|0.00|0.22|0.00|0.18|20|-42|-42|20,20,42,20,31.0,NaN
1,chr17-31229332-A-C,0.00,-78,-0.00,-1,missense_variant,c.2717A>C,p.His906Pro,0.01,-78,-0.00,-100,C|NF1|0.01|0.00|0.01|0.00|-78|-307|-11|-100,133,307,133,21.0,NaN
2,chr17-31326205-C-A,0.02,47,-0.01,-7,missense_variant,c.5221C>A,p.His1741Asn,0.00,-92,-0.02,-117,A|NF1|0.00|0.00|0.04|0.02|-92|123|-7|-117,47,385,47,37.0,Conflicting_classifications_of_pathogenicity
3,chr17-31337875-T-G,0.01,5,-0.00,264,synonymous_variant,c.6699T>G,p.Ala2233%3D,0.00,-392,0.00,5,G|NF1|0.00|0.00|0.00|0.00|-392|150|5|264,5,56,5,44.0,NaN
4,chr17-31206272-T-C,0.00,-32,-0.00,17,synonymous_variant,c.1293T>C,p.Ala431%3D,0.00,-32,0.00,-161,C|NF1|0.00|0.00|0.00|0.00|-32|192|-161|229,32,32,99,12.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29035,chr17-31226542-T-G,0.01,-107,-0.00,41,synonymous_variant,c.2109T>G,p.Val703%3D,0.00,-107,0.00,142,G|NF1|0.00|0.01|0.00|0.00|-107|44|142|68,107,107,142,18.0,NaN
29036,chr17-31343137-T-G,0.13,87,-0.88,-2,splice_donor_variant&intron_variant,c.7189+2T>G,p.?,0.00,-1,-1.00,-2,G|NF1|0.00|0.69|0.08|1.00|-1|-128|87|-2,0,0,0,NaN,NaN
29037,chr17-31340514-C-T,0.01,70,-0.01,-9,synonymous_variant,c.6931C>T,p.Leu2311%3D,0.03,70,-0.03,69,T|NF1|0.03|0.00|0.00|0.03|70|-9|-348|69,9,9,131,47.0,NaN
29038,chr17-31350217-C-A,0.00,-22,-0.00,101,synonymous_variant,c.7356C>A,p.Arg2452%3D,0.00,-22,0.00,227,A|NF1|0.00|0.00|0.00|0.00|-22|6|227|-132,34,34,101,50.0,NaN


In [ ]:
# add additional columns
library["chrom"] = [x.split('-')[0] for x in library["var_id"]]
library["start"] = [int(x.split('-')[1]) for x in library["var_id"]]
library["ref"] = [x.split('-')[2] for x in library["var_id"]]
library["len_ref"] = [len(x) for x in library["ref"]]
library["alt"] = [x.split('-')[3] for x in library["var_id"]]
library["len_alt"] = [len(x) for x in library["alt"]]
library["end"] = library["start"] +  library["len_ref"] - 1
library["splice_effect_to_canon_ss_dist"] = np.minimum(abs(library["pangolin_gain_pos"] + library["distance_to_ss_left"]), abs(library["pangolin_gain_pos"] - library["distance_to_ss_right"])) # should not be close to 0

# filter the library
library = library[~library["exon_number"].isna()] # filter for only exonic
library = library[~(library["ref"].str.contains("N") | library["alt"].str.contains("N"))] # filter variants with unknown bases
library.loc[library['clinvar'].isna(), 'clinvar'] = "unknown"# fill na values in clinvar column
library = library[library["exon_number"].isin([17,25,34,39,42,27])] # filter for exons as determined elsewhere
library

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
21,chr17-31330315-C-A,0.10,107,-0.17,-19,missense_variant,c.5629C>A,p.Leu1877Met,0.09,107,...,39.0,Likely_pathogenic,chr17,31330315,C,1,A,1,31330315,76
33,chr17-31232085-G-C,0.01,-12,-0.02,104,missense_variant,c.3210G>C,p.Gln1070His,0.01,-12,...,25.0,unknown,chr17,31232085,G,1,C,1,31232085,0
34,chr17-31260477-C-G,0.01,1,-0.01,-108,missense_variant,c.4539C>G,p.Asn1513Lys,0.04,-39,...,34.0,Uncertain_significance,chr17,31260477,C,1,G,1,31260477,37
37,chr17-31233115-C-G,0.01,-234,-0.05,-113,missense_variant,c.3610C>G,p.Arg1204Gly,0.00,-376,...,27.0,Pathogenic/Likely_pathogenic,chr17,31233115,C,1,G,1,31233115,121
45,chr17-31330338-T-G,0.06,-42,-0.09,84,missense_variant,c.5652T>G,p.Phe1884Leu,0.09,-42,...,39.0,Pathogenic/Likely_pathogenic,chr17,31330338,T,1,G,1,31330338,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28975,chr17-31225145-T-G,0.02,-4,-0.12,-50,missense_variant,c.1896T>G,p.Cys632Trp,0.00,-3,...,17.0,unknown,chr17,31225145,T,1,G,1,31225145,46
28993,chr17-31225117-A-G,0.12,-22,-0.14,24,missense_variant,c.1868A>G,p.His623Arg,0.08,-19,...,17.0,Conflicting_classifications_of_pathogenicity,chr17,31225117,A,1,G,1,31225117,0
28994,chr17-31233189-C-G,0.00,24,-0.01,-489,synonymous_variant,c.3684C>G,p.Ala1228%3D,0.00,-187,...,27.0,unknown,chr17,31233189,C,1,G,1,31233189,0
29017,chr17-31330406-T-A,0.04,-110,-0.12,16,missense_variant,c.5720T>A,p.Phe1907Tyr,0.07,-110,...,39.0,Uncertain_significance,chr17,31330406,T,1,A,1,31330406,0


In [ ]:
positions_oi = library.loc[((library["pangolin_gain"] > 0.2) & (abs(library["pangolin_loss"]) < abs(library["pangolin_gain"])) & (library["splice_effect_to_canon_ss_dist"] > 10)), "start"].unique()
ss_lib = library[library["start"].isin(positions_oi)]
ss_lib = ss_lib[(ss_lib["clinvar"] != "unknown") | (ss_lib["pangolin_gain"] > 0.2)]

# taken from ex27 screen
# make sure to add these to the library
ex27ss_vars = [
    "chr17-31233072-A-G",
    "chr17-31233123-G-A",
    "chr17-31233156-T-G", #
    "chr17-31233126-A-G"
]
ss_lib = library[library["var_id"].isin(set(ss_lib["var_id"]) | set(ex27ss_vars))]
ss_lib

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
512,chr17-31225227-A-T,0.88,-2,-0.74,23,stop_gained,c.1978A>T,p.Lys660Ter,0.06,-129,...,17.0,unknown,chr17,31225227,A,1,T,1,31225227,25
549,chr17-31260369-G-T,0.49,17,-0.34,0,missense_variant&splice_region_variant,c.4431G>T,p.Arg1477Ser,0.49,17,...,34.0,Uncertain_significance,chr17,31260369,G,1,T,1,31260369,17
682,chr17-31225134-G-A,0.82,2,-0.73,-39,missense_variant,c.1885G>A,p.Gly629Arg,0.99,2,...,17.0,Pathogenic,chr17,31225134,G,1,A,1,31225134,41
990,chr17-31225238-G-T,0.88,-2,-0.82,12,synonymous_variant,c.1989G>T,p.Gly663%3D,0.05,-140,...,17.0,Uncertain_significance,chr17,31225238,G,1,T,1,31225238,14
1017,chr17-31232180-T-G,0.04,9,-0.05,9,stop_gained,c.3305T>G,p.Leu1102Ter,0.00,-46,...,25.0,Pathogenic,chr17,31232180,T,1,G,1,31232180,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28234,chr17-31232155-G-T,0.71,-6,-0.37,34,stop_gained,c.3280G>T,p.Glu1094Ter,0.01,-80,...,25.0,Pathogenic,chr17,31232155,G,1,T,1,31232155,40
28378,chr17-31232145-A-T,0.66,-2,-0.14,44,synonymous_variant,c.3270A>T,p.Gly1090%3D,0.01,-70,...,25.0,Uncertain_significance,chr17,31232145,A,1,T,1,31232145,46
28405,chr17-31232164-G-T,0.85,-2,-0.47,25,stop_gained,c.3289G>T,p.Glu1097Ter,0.07,-89,...,25.0,unknown,chr17,31232164,G,1,T,1,31232164,27
28505,chr17-31232165-A-T,0.71,-2,-0.29,24,missense_variant,c.3290A>T,p.Glu1097Val,0.00,-382,...,25.0,unknown,chr17,31232165,A,1,T,1,31232165,26


In [201]:
# extract stop variants
stop_gained_lib = library[library["consequence"] == "stop_gained"]
stop_gained_lib = stop_gained_lib[(stop_gained_lib["pangolin_gain"] < 0.1) & (stop_gained_lib["distance_to_ss"] >= 3)] # remove overlapping stop codon AND splicing predicted
stop_gained_lib

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
967,chr17-31232174-C-A,0.03,-25,-0.17,15,stop_gained,c.3299C>A,p.Ser1100Ter,0.00,-391,...,25.0,Pathogenic,chr17,31232174,C,1,A,1,31232174,40
1017,chr17-31232180-T-G,0.04,9,-0.05,9,stop_gained,c.3305T>G,p.Leu1102Ter,0.00,-46,...,25.0,Pathogenic,chr17,31232180,T,1,G,1,31232180,0
1058,chr17-31330483-G-T,0.01,190,-0.34,15,stop_gained,c.5797G>T,p.Gly1933Ter,0.01,88,...,39.0,unknown,chr17,31330483,G,1,T,1,31330483,175
1273,chr17-31330379-T-A,0.04,43,-0.09,-83,stop_gained,c.5693T>A,p.Leu1898Ter,0.03,43,...,39.0,unknown,chr17,31330379,T,1,A,1,31330379,76
1670,chr17-31336793-C-A,0.01,26,-0.23,-158,stop_gained,c.6306C>A,p.Tyr2102Ter,0.06,26,...,42.0,Pathogenic,chr17,31336793,C,1,A,1,31336793,95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27812,chr17-31260458-T-A,0.01,26,-0.09,57,stop_gained,c.4520T>A,p.Leu1507Ter,0.07,-20,...,34.0,unknown,chr17,31260458,T,1,A,1,31260458,31
28341,chr17-31336693-T-A,0.00,15,-0.14,-58,stop_gained,c.6206T>A,p.Leu2069Ter,0.00,-353,...,42.0,Pathogenic,chr17,31336693,T,1,A,1,31336693,73
28489,chr17-31336897-C-G,0.03,-262,-0.01,17,stop_gained,c.6410C>G,p.Ser2137Ter,0.01,-262,...,42.0,Pathogenic,chr17,31336897,C,1,G,1,31336897,0
28886,chr17-31260458-T-G,0.03,-20,-0.07,57,stop_gained,c.4520T>G,p.Leu1507Ter,0.17,-20,...,34.0,Pathogenic,chr17,31260458,T,1,G,1,31260458,69


In [ ]:
possible_syn_muts = {}

for var_ind in stop_gained_lib.index:
    var_genom_pos = int(stop_gained_lib["start"][var_ind])
    var_ref_base = stop_gained_lib["ref"][var_ind]
    var_alt_base = stop_gained_lib["alt"][var_ind]
    var_chrom = stop_gained_lib["chrom"][var_ind]
    var_id = stop_gained_lib["var_id"][var_ind]
    mut_oi = [var_chrom, var_genom_pos, var_ref_base, var_alt_base]

    possible_syn_muts[var_id] = []

    # filter for snvs
    if len(var_ref_base) != 1 or len(var_alt_base) != 1:
        continue

    var_cdna_pos = functions.genomic2cdna(exons_oi, var_genom_pos)
    #print(var_cdna_pos)

    if var_cdna_pos is None:
        #print("INTRONIC VARIANT!")
        continue

    codon_position = functions.cdna2codon(var_cdna_pos)
    #print(codon_position)

    # compute codon start position
    codon_start = functions.compute_codon_start(var_genom_pos, codon_position)
    #print(codon_start)

    # compute synonymous mutations
    exon_oi = exons_oi[(exons_oi["start"] <= var_genom_pos) & (var_genom_pos <= exons_oi["end"])]
    exon_oi = exon_oi.reset_index()
    exon_start = exon_oi.loc[0,"start"]
    exon_end = exon_oi.loc[0,"end"]


    is_before_variant = codon_start <= var_genom_pos and codon_start >= exon_start # synonymous variant is before the variant of interest
    is_after_variant = codon_start > var_genom_pos and  codon_start+2 <= exon_end # synonymous variant is after the variant of interest
    if is_before_variant or is_after_variant: 
        codon = genome.fetch(var_chrom, codon_start - 1, codon_start + 2)
        new_syn_muts = functions.get_synonymous_variants(codon, var_chrom, codon_start)
        possible_syn_muts[var_id] = new_syn_muts

In [203]:
# convert to dataframe and filter syn vars which have a splicing effect
possible_syn_muts_df = []
for stop_var in possible_syn_muts:
    syn_vars = possible_syn_muts[stop_var]
    for syn_var in syn_vars:
        stop_var_parts = stop_var.split('-')
        stop_var_chrom = stop_var_parts[0]
        stop_var_pos = stop_var_parts[1]
        stop_var_ref = stop_var_parts[2]
        stop_var_alt = stop_var_parts[3]
        possible_syn_muts_df.append({
            "var_id": '-'.join([str(x) for x in syn_var]),
            "stop_var_id": stop_var,
            "stop_var_chrom": stop_var_chrom,
            "stop_var_pos": stop_var_pos,
            "stop_var_ref": stop_var_ref,
            "stop_var_alt": stop_var_alt
        })
possible_syn_muts_df = pd.DataFrame(possible_syn_muts_df)
possible_syn_muts_df = possible_syn_muts_df.merge(library, on="var_id", how = "inner")
##possible_syn_muts_df = possible_syn_muts_df.groupby("stop_var_pos").apply(lambda x: x.sort_values("pangolin_gain").iloc[0]).reset_index(drop = True)
possible_syn_muts_df = possible_syn_muts_df[possible_syn_muts_df["pangolin_gain"] < 0.1]
possible_syn_muts_df

,var_id,stop_var_id,stop_var_chrom,stop_var_pos,stop_var_ref,stop_var_alt,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
0,chr17-31232175-A-T,chr17-31232174-C-A,chr17,31232174,C,A,0.02,-26,-0.07,14,...,25.0,unknown,chr17,31232175,A,1,T,1,31232175,40
1,chr17-31232175-A-C,chr17-31232174-C-A,chr17,31232174,C,A,0.03,14,-0.05,2,...,25.0,unknown,chr17,31232175,A,1,C,1,31232175,0
2,chr17-31232175-A-G,chr17-31232174-C-A,chr17,31232174,C,A,0.09,14,-0.06,2,...,25.0,Likely_benign,chr17,31232175,A,1,G,1,31232175,0
3,chr17-31232181-A-G,chr17-31232180-T-G,chr17,31232180,T,G,0.01,277,-0.08,-4,...,25.0,unknown,chr17,31232181,A,1,G,1,31232181,269
5,chr17-31330485-A-T,chr17-31330483-G-T,chr17,31330483,G,T,0.00,188,-0.15,-189,...,39.0,unknown,chr17,31330485,A,1,T,1,31330485,175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,chr17-31336898-A-T,chr17-31336897-C-G,chr17,31336897,C,G,0.01,-71,-0.06,16,...,42.0,unknown,chr17,31336898,A,1,T,1,31336898,87
163,chr17-31336898-A-C,chr17-31336897-C-G,chr17,31336897,C,G,0.01,16,-0.03,16,...,42.0,unknown,chr17,31336898,A,1,C,1,31336898,0
165,chr17-31260459-A-G,chr17-31260458-T-G,chr17,31260458,T,G,0.01,-21,-0.00,-73,...,34.0,unknown,chr17,31260459,A,1,G,1,31260459,69
166,chr17-31260457-T-C,chr17-31260458-T-G,chr17,31260458,T,G,0.02,-19,-0.00,27,...,34.0,unknown,chr17,31260457,T,1,C,1,31260457,69


In [205]:
# make sure that we can find at least one synonymous variant at the codon position of the stop codon
stops_with_syn_muts = list(possible_syn_muts_df["stop_var_id"].unique()) #[x for x in possible_syn_muts if len(possible_syn_muts[x]) > 0]
stop_gained_lib = stop_gained_lib[stop_gained_lib["var_id"].isin(stops_with_syn_muts)]
stop_gained_lib

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
967,chr17-31232174-C-A,0.03,-25,-0.17,15,stop_gained,c.3299C>A,p.Ser1100Ter,0.00,-391,...,25.0,Pathogenic,chr17,31232174,C,1,A,1,31232174,40
1017,chr17-31232180-T-G,0.04,9,-0.05,9,stop_gained,c.3305T>G,p.Leu1102Ter,0.00,-46,...,25.0,Pathogenic,chr17,31232180,T,1,G,1,31232180,0
1058,chr17-31330483-G-T,0.01,190,-0.34,15,stop_gained,c.5797G>T,p.Gly1933Ter,0.01,88,...,39.0,unknown,chr17,31330483,G,1,T,1,31330483,175
1273,chr17-31330379-T-A,0.04,43,-0.09,-83,stop_gained,c.5693T>A,p.Leu1898Ter,0.03,43,...,39.0,unknown,chr17,31330379,T,1,A,1,31330379,76
1670,chr17-31336793-C-A,0.01,26,-0.23,-158,stop_gained,c.6306C>A,p.Tyr2102Ter,0.06,26,...,42.0,Pathogenic,chr17,31336793,C,1,A,1,31336793,95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27812,chr17-31260458-T-A,0.01,26,-0.09,57,stop_gained,c.4520T>A,p.Leu1507Ter,0.07,-20,...,34.0,unknown,chr17,31260458,T,1,A,1,31260458,31
28341,chr17-31336693-T-A,0.00,15,-0.14,-58,stop_gained,c.6206T>A,p.Leu2069Ter,0.00,-353,...,42.0,Pathogenic,chr17,31336693,T,1,A,1,31336693,73
28489,chr17-31336897-C-G,0.03,-262,-0.01,17,stop_gained,c.6410C>G,p.Ser2137Ter,0.01,-262,...,42.0,Pathogenic,chr17,31336897,C,1,G,1,31336897,0
28886,chr17-31260458-T-G,0.03,-20,-0.07,57,stop_gained,c.4520T>G,p.Leu1507Ter,0.17,-20,...,34.0,Pathogenic,chr17,31260458,T,1,G,1,31260458,69


In [206]:
# manually select stop codons for exon 27 from previous screen
#chr17-31233016-A-T 29560034
# GTTGATCCGTCTCAACCGATTAGGCTTAGGTTACCACAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTTGTCTGGAGgTCCTaGTGGTAACCTAAGCCTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCTAGGACCTCCAGACAAGAGCGGGAGAGACGGTTCC
#chr17-31233128-T-A 29560146
# GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTGACCAGTTCaACCtATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC
#chr17-31233005-T-G 29560023
# GTTGATCCGTCTCAACCGTGTTTGTCCACATTAGGCTTGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCCTTGTGGTAgCCTcAGCCTAATGTGGACAATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCATTAGGCTGAGGCTACCACCGGGAGAGACGGTTCC
#chr17-31233157-C-T 29560175
# GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTAGGGAGTTCcCCTTaATCACCCATCATTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAAGGCACAGAATTTGCGGGAGAGACGGTTCC
#chr17-31233133-G-T 29560151
# GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCATTGTGACaAGTTaCACCAATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC
ex27stop_vars = ["chr17-31233016-A-T", "chr17-31233128-T-A", "chr17-31233005-T-G", "chr17-31233157-C-T", "chr17-31233133-G-T"]
ex27stop_vars = stop_gained_lib[stop_gained_lib["var_id"].isin(ex27stop_vars)]
assert len(ex27stop_vars) == 5

stop_gained_lib = stop_gained_lib[~(stop_gained_lib["exon_number"] == 27)]
stop_gained_lib

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
967,chr17-31232174-C-A,0.03,-25,-0.17,15,stop_gained,c.3299C>A,p.Ser1100Ter,0.00,-391,...,25.0,Pathogenic,chr17,31232174,C,1,A,1,31232174,40
1017,chr17-31232180-T-G,0.04,9,-0.05,9,stop_gained,c.3305T>G,p.Leu1102Ter,0.00,-46,...,25.0,Pathogenic,chr17,31232180,T,1,G,1,31232180,0
1058,chr17-31330483-G-T,0.01,190,-0.34,15,stop_gained,c.5797G>T,p.Gly1933Ter,0.01,88,...,39.0,unknown,chr17,31330483,G,1,T,1,31330483,175
1273,chr17-31330379-T-A,0.04,43,-0.09,-83,stop_gained,c.5693T>A,p.Leu1898Ter,0.03,43,...,39.0,unknown,chr17,31330379,T,1,A,1,31330379,76
1670,chr17-31336793-C-A,0.01,26,-0.23,-158,stop_gained,c.6306C>A,p.Tyr2102Ter,0.06,26,...,42.0,Pathogenic,chr17,31336793,C,1,A,1,31336793,95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27759,chr17-31330383-T-A,0.04,39,-0.08,-87,stop_gained,c.5697T>A,p.Cys1899Ter,0.01,-29,...,39.0,Pathogenic,chr17,31330383,T,1,A,1,31330383,76
27812,chr17-31260458-T-A,0.01,26,-0.09,57,stop_gained,c.4520T>A,p.Leu1507Ter,0.07,-20,...,34.0,unknown,chr17,31260458,T,1,A,1,31260458,31
28341,chr17-31336693-T-A,0.00,15,-0.14,-58,stop_gained,c.6206T>A,p.Leu2069Ter,0.00,-353,...,42.0,Pathogenic,chr17,31336693,T,1,A,1,31336693,73
28489,chr17-31336897-C-G,0.03,-262,-0.01,17,stop_gained,c.6410C>G,p.Ser2137Ter,0.01,-262,...,42.0,Pathogenic,chr17,31336897,C,1,G,1,31336897,0


In [207]:
# select 5 random stop codon variants per exon
seed = 235
def get_random(items: list, n: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    result = np.array(list(items))
    rng.shuffle(result)
    return result[0:n]

stop_gained_lib = stop_gained_lib.groupby("start").apply(lambda x: x.sort_values("pangolin_gain").iloc[0]).reset_index(drop = True) # select one random per start position such that we do not have multiple stop codons at one position
stop_gained_lib = stop_gained_lib.groupby("exon_number").apply(lambda x: x[x["var_id"].isin(get_random(x["var_id"], 5, seed))]).reset_index(drop = True) # select 5 random stop variants per exon
stop_gained_lib = pd.concat([stop_gained_lib, ex27stop_vars]) # add ex27 back again
stop_gained_lib

/tmp/ipykernel_3362068/481535777.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stop_gained_lib = stop_gained_lib.groupby("start").apply(lambda x: x.sort_values("pangolin_gain").iloc[0]).reset_index(drop = True) # select one random per start position such that we do not have multiple stop codons at one position
/tmp/ipykernel_3362068/481535777.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stop_gained_lib 

,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,hgvsc,hgvsp,spliceai_gain,spliceai_gain_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
0,chr17-31225145-T-A,0.00,-1,-0.21,105,stop_gained,c.1896T>A,p.Cys632Ter,0.00,2,...,17.0,unknown,chr17,31225145,T,1,A,1,31225145,49
1,chr17-31225173-C-T,0.01,-37,-0.31,77,stop_gained,c.1924C>T,p.Gln642Ter,0.00,77,...,17.0,Pathogenic,chr17,31225173,C,1,T,1,31225173,41
2,chr17-31225198-T-A,0.01,-57,-0.24,52,stop_gained,c.1949T>A,p.Leu650Ter,0.00,-38,...,17.0,Pathogenic,chr17,31225198,T,1,A,1,31225198,46
3,chr17-31225230-G-T,0.01,-89,-0.03,20,stop_gained,c.1981G>T,p.Gly661Ter,0.02,-132,...,17.0,unknown,chr17,31225230,G,1,T,1,31225230,46
4,chr17-31225233-A-T,0.01,41,-0.23,17,stop_gained,c.1984A>T,p.Lys662Ter,0.00,17,...,17.0,Pathogenic,chr17,31225233,A,1,T,1,31225233,24
5,chr17-31232083-C-T,0.01,3,-0.13,-10,stop_gained,c.3208C>T,p.Gln1070Ter,0.00,-406,...,25.0,Pathogenic,chr17,31232083,C,1,T,1,31232083,13
6,chr17-31232108-C-G,0.01,41,-0.05,81,stop_gained,c.3233C>G,p.Ser1078Ter,0.00,-431,...,25.0,Pathogenic,chr17,31232108,C,1,G,1,31232108,40
7,chr17-31232170-A-T,0.02,-21,-0.11,19,stop_gained,c.3295A>T,p.Lys1099Ter,0.00,-387,...,25.0,unknown,chr17,31232170,A,1,T,1,31232170,40
8,chr17-31232176-C-T,0.05,-27,-0.25,13,stop_gained,c.3301C>T,p.Gln1101Ter,0.00,-499,...,25.0,Pathogenic,chr17,31232176,C,1,T,1,31232176,40
9,chr17-31232180-T-G,0.04,9,-0.05,9,stop_gained,c.3305T>G,p.Leu1102Ter,0.00,-46,...,25.0,Pathogenic,chr17,31232180,T,1,G,1,31232180,0


In [208]:
syn_mut_lib = possible_syn_muts_df[possible_syn_muts_df["stop_var_id"].isin(stop_gained_lib["var_id"])]
syn_mut_lib

,var_id,stop_var_id,stop_var_chrom,stop_var_pos,stop_var_ref,stop_var_alt,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,...,exon_number,clinvar,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist
3,chr17-31232181-A-G,chr17-31232180-T-G,chr17,31232180,T,G,0.01,277,-0.08,-4,...,25.0,unknown,chr17,31232181,A,1,G,1,31232181,269
11,chr17-31232109-A-T,chr17-31232108-C-G,chr17,31232108,C,G,0.01,80,-0.01,40,...,25.0,Likely_benign,chr17,31232109,A,1,T,1,31232109,0
12,chr17-31232109-A-C,chr17-31232108-C-G,chr17,31232108,C,G,0.00,80,-0.01,40,...,25.0,Likely_benign,chr17,31232109,A,1,C,1,31232109,0
13,chr17-31232109-A-G,chr17-31232108-C-G,chr17,31232108,C,G,0.01,-36,-0.02,80,...,25.0,unknown,chr17,31232109,A,1,G,1,31232109,0
16,chr17-31330378-T-C,chr17-31330379-T-G,chr17,31330379,T,G,0.02,-82,-0.04,44,...,39.0,unknown,chr17,31330378,T,1,C,1,31330378,0
17,chr17-31225145-T-C,chr17-31225145-T-A,chr17,31225145,T,A,0.04,-4,-0.00,15,...,17.0,unknown,chr17,31225145,T,1,C,1,31225145,46
22,chr17-31233129-G-A,chr17-31233128-T-A,chr17,31233128,T,A,0.01,-248,-0.02,-127,...,27.0,Likely_benign,chr17,31233129,G,1,A,1,31233129,121
23,chr17-31233127-T-C,chr17-31233128-T-A,chr17,31233128,T,A,0.01,86,-0.01,-246,...,27.0,Likely_benign,chr17,31233127,T,1,C,1,31233127,0
36,chr17-31225232-A-C,chr17-31225230-G-T,chr17,31225230,G,T,0.01,-137,-0.01,-91,...,17.0,unknown,chr17,31225232,A,1,C,1,31225232,0
37,chr17-31225232-A-G,chr17-31225230-G-T,chr17,31225230,G,T,0.01,42,-0.04,18,...,17.0,Likely_benign,chr17,31225232,A,1,G,1,31225232,24


In [ ]:
# save library to file
library.to_csv(basepath + "/search_sequences/library_design/design_sequences/data/variants.tsv", sep = "\t", header = True, index = False)

compute synonymous marker mutations

In [ ]:
# get cdna position of variant
outfile = open(basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_syn_muts.vcf", "w")

#library_oi = ss_lib
#library_oi = stop_gained_lib
library_oi = syn_mut_lib


header_lines = [
    '##fileformat=VCFv4.1',
    '##fileDate=2025-08-31',
    '##reference=GRCh38',
    '##contig=<ID=chr1>',
    '##contig=<ID=chr2>',
    '##contig=<ID=chr3>',
    '##contig=<ID=chr4>',
    '##contig=<ID=chr5>',
    '##contig=<ID=chr6>',
    '##contig=<ID=chr7>',
    '##contig=<ID=chr8>',
    '##contig=<ID=chr9>',
    '##contig=<ID=chr10>',
    '##contig=<ID=chr11>',
    '##contig=<ID=chr12>',
    '##contig=<ID=chr13>',
    '##contig=<ID=chr14>',
    '##contig=<ID=chr15>',
    '##contig=<ID=chr16>',
    '##contig=<ID=chr17>',
    '##contig=<ID=chr18>',
    '##contig=<ID=chr19>',
    '##contig=<ID=chr20>',
    '##contig=<ID=chr21>',
    '##contig=<ID=chr22>',
    '##contig=<ID=chrX>',
    '##contig=<ID=chrY>',
    '##ID=<Description="Combined splice variant plus synonymous marker variant">',
    '##INFO=<ID=mut_var,Number=1,Type=String,Description="The splice variant of interest">',
    '##INFO=<ID=syn_var,Number=1,Type=String,Description="The synonymous marker variant">',
    '#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO'
]

for line in header_lines:
    outfile.write(line + "\n")


for var_ind in library_oi.index:
    var_genom_pos = int(library_oi["start"][var_ind])
    var_ref_base = library_oi["ref"][var_ind]
    var_alt_base = library_oi["alt"][var_ind]
    var_chrom = library_oi["chrom"][var_ind]
    mut_oi = [var_chrom, var_genom_pos, var_ref_base, var_alt_base]

    # filter for snvs
    if len(var_ref_base) != 1 or len(var_alt_base) != 1:
        continue

    var_cdna_pos = functions.genomic2cdna(exons_oi, var_genom_pos)
    #print(var_cdna_pos)

    if var_cdna_pos is None:
        #print("INTRONIC VARIANT!")
        continue

    codon_position = functions.cdna2codon(var_cdna_pos)
    #print(codon_position)

    # compute adjacent codon start positions
    codon_starts_oi = functions.compute_adjacent_codons(var_genom_pos, codon_position, n_codons=3, verbose = False)
    #print(next_codon_start)
    #print(prev_codon_start)

    # compute synonymous mutations
    exon_oi = exons_oi[(exons_oi["start"] <= var_genom_pos) & (var_genom_pos <= exons_oi["end"])]
    exon_oi = exon_oi.reset_index()
    exon_start = exon_oi.loc[0,"start"]
    exon_end = exon_oi.loc[0,"end"]

    req_muts = []
    for codon_start in codon_starts_oi:
        is_before_variant = codon_start < var_genom_pos and codon_start >= exon_start # synonymous variant is before the variant of interest
        is_after_variant = codon_start > var_genom_pos and  codon_start+2 <= exon_end # synonymous variant is after the variant of interest
        if is_before_variant or is_after_variant: 
            codon = genome.fetch(var_chrom, codon_start - 1, codon_start + 2)
            new_syn_muts = functions.get_synonymous_variants(codon, var_chrom, codon_start)
            req_muts.extend(new_syn_muts)

    # get vcf
    for syn_mut in req_muts:
        combined_variant = functions.get_mutated_sequence([mut_oi,syn_mut], genome, format = "alt")

        info_parts = [
            "mut_var=" + '-'.join([str(x) for x in mut_oi]),
            "syn_var=" + '-'.join([str(x) for x in syn_mut]),
        ]
        vcf_data = [combined_variant[0], str(combined_variant[1]), ".", combined_variant[2], combined_variant[3], ".", ".", ';'.join(info_parts)]
        new_line = '\t'.join(vcf_data)
        outfile.write(new_line + "\n")

outfile.close()

ANALYZE variants with synonymous markers

In [ ]:
def read_combined_variants_data(path, full_library):
    combined_variants_data = []
    with open(path, "r") as combined_variants_pangolin:
        for line in combined_variants_pangolin:
            if line.startswith('#') or line.strip() == '':
                continue

            parts = line.split('\t')
            ##CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
            chrom = parts[0]
            pos = int(parts[1])
            ref = parts[3]
            alt = parts[4]
            info = parts[7]

            mut_var = functions.find_between(info, "mut_var=", "(;|$)")
            syn_var = functions.find_between(info, "syn_var=", "(;|$)")

            pangolin = functions.find_between(info, "Pangolin=", "(;|$)") # ENSG00000196712.18|-5:0.8100000023841858|25:-0.6399999856948853|Warnings:
            pangolin_parts = pangolin.split('|')
            pangolin_gain = pangolin_parts[1].split(':')
            pangolin_loss = pangolin_parts[2].split(':')

            distance_to_ss = 0
            distance_to_ss_left = 0
            distance_to_ss_right = 0
            for current_exon_number, exon_boundary in enumerate(exon_boundaries_oi):
                if exon_boundary[0] <= pos <= exon_boundary[1]:
                    distance_to_ss_left = pos - exon_boundary[0]
                    distance_to_ss_right = exon_boundary[1] - pos
                    distance_to_ss = min(distance_to_ss_left, distance_to_ss_right)

            new_data = {
                "var_id": mut_var,
                "syn_var": syn_var,
                "start_combined": pos,
                "pangolin_gain_score_combined": round(float(pangolin_gain[1]), 2),
                "pangolin_gain_pos_combined": int(pangolin_gain[0]),
                "pangolin_loss_score_combined": round(float(pangolin_loss[1]), 2),
                "pangolin_loss_pos_combined": int(pangolin_loss[0]),
                "distance_to_ss_combined": distance_to_ss,
                "distance_to_ss_left_combined": distance_to_ss_left,
                "distance_to_ss_right_combined": distance_to_ss_right,
            }
            combined_variants_data.append(new_data)
    combined_variants_data = pd.DataFrame(combined_variants_data)

    # add additional columns
    combined_variants_data = combined_variants_data.merge(full_library, how = "left", on="var_id")
    combined_variants_data["pangolin_gain_pos_dist"] = abs(abs(combined_variants_data["pangolin_gain_pos_combined"] - combined_variants_data["pangolin_gain_pos"]) - abs(combined_variants_data["start"] - combined_variants_data["start_combined"]))
    combined_variants_data["pangolin_gain_score_dist"] = abs(combined_variants_data["pangolin_gain_score_combined"] - combined_variants_data["pangolin_gain"])
    combined_variants_data["pangolin_gain_combined_score"] = combined_variants_data["pangolin_gain_pos_dist"] + (combined_variants_data["pangolin_gain_score_dist"] * 100)
    combined_variants_data["splice_effect_to_canon_ss_dist_combined"] = np.minimum(abs(combined_variants_data["pangolin_gain_pos_combined"] + combined_variants_data["distance_to_ss_left_combined"]), abs(combined_variants_data["pangolin_gain_pos_combined"] - combined_variants_data["distance_to_ss_right_combined"])) # should not be close to 0
    combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)
    combined_variants_data.loc[combined_variants_data['clinvar'].isna(), 'clinvar'] = "unknown"
    combined_variants_data["pangolin_gain_combined_bin"] = [decide_bin(x) for x in combined_variants_data["pangolin_gain_score_combined"]]
    combined_variants_data["pangolin_gain_bin"] = [decide_bin(x) for x in combined_variants_data["pangolin_gain"]]
    combined_variants_data = combined_variants_data[~combined_variants_data["exon_number"].isna()]
    combined_variants_data["is_clinvar"] = combined_variants_data["clinvar"] != "unknown"
    combined_variants_data["is_pangolin"] = combined_variants_data["pangolin_gain_bin"] != "no_splicing"
    
    return combined_variants_data

splice variant library with 2 good marker variants

In [ ]:
# read data
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_ss_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, library)

# filter for ss variants in specific exons 
test = library[library["exon_number"].isin([17,25,34,39,42,27])]
test = test[(test["pangolin_gain"] > 0.2) & (abs(test["pangolin_loss"]) < abs(test["pangolin_gain"])) & (test["splice_effect_to_canon_ss_dist"] > 10) | (test["start"] == 31233123)]
positions_oi = test["start"].unique()
print(len(positions_oi))

combined_variants_data = combined_variants_data[combined_variants_data["start"].isin(positions_oi)]
print(len(combined_variants_data["start"].unique()))

# require the splice variant to create a new ss which is at least 10 bp away from the closest canonical ss and allow no_splicing variants because they are from ClinVAr
# remove marker variants which push the predicted ss to the canonical ss. Assumes that the variant_oi itself does not influence the canonical ss.
preferred_synonymous_variants = combined_variants_data[(combined_variants_data["splice_effect_to_canon_ss_dist_combined"] >= 10) | (combined_variants_data["pangolin_gain_bin"] == "no_splicing")]

# remove variants with loss > gain
vars2remove = preferred_synonymous_variants[preferred_synonymous_variants["is_pangolin"]][~preferred_synonymous_variants[preferred_synonymous_variants["is_pangolin"]]["var_id"].isin(test["var_id"])]["var_id"]
preferred_synonymous_variants = preferred_synonymous_variants[~preferred_synonymous_variants["var_id"].isin(vars2remove)]

# order by combined_score and select the top x with the lowest scores (strictly speaking this is more a distance than a score) -> these influence spliceing the least
take_top_markers = 2
preferred_synonymous_variants = preferred_synonymous_variants.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True).iloc[0:take_top_markers]) # take top x
preferred_synonymous_variants = preferred_synonymous_variants.reset_index(drop=True)

final_ss_lib = preferred_synonymous_variants.copy()
final_ss_lib["source"] = "ss_lib"
final_ss_lib

76
76


/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)
/tmp/ipykernel_3362068/3028472195.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  preferred_synonymous_variants = preferred_synonymous_variants.groupby("var_id")

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,splice_effect_to_canon_ss_dist,pangolin_gain_pos_dist,pangolin_gain_score_dist,pangolin_gain_combined_score,splice_effect_to_canon_ss_dist_combined,pangolin_gain_combined_bin,pangolin_gain_bin,is_clinvar,is_pangolin,source
0,chr17-31225098-G-A,chr17-31225109-T-C,31225098,0.07,43,-0.04,-3,3,3,152,...,46,0,0.01,1.0,46,no_splicing,no_splicing,True,False,ss_lib
1,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.10,43,-0.18,152,3,3,152,...,46,0,0.02,2.0,46,no_splicing,no_splicing,True,False,ss_lib
2,chr17-31225098-G-T,chr17-31225109-T-C,31225098,0.19,43,-0.17,0,3,3,152,...,46,0,0.03,3.0,46,no_splicing,low_gain,False,True,ss_lib
3,chr17-31225098-G-T,chr17-31225106-A-G,31225098,0.18,43,-0.29,152,3,3,152,...,46,0,0.04,4.0,46,no_splicing,low_gain,False,True,ss_lib
4,chr17-31225100-A-T,chr17-31225109-T-C,31225100,0.20,41,-0.04,-5,5,5,150,...,46,0,0.01,1.0,46,no_splicing,low_gain,False,True,ss_lib
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,chr17-31336865-C-G,chr17-31336868-T-A,31336865,0.08,49,-0.02,49,49,230,49,...,0,0,0.03,3.0,0,no_splicing,no_splicing,True,False,ss_lib
256,chr17-31336865-C-T,chr17-31336856-T-C,31336856,0.01,-29,-0.05,58,58,221,58,...,87,0,0.00,0.0,87,no_splicing,no_splicing,True,False,ss_lib
257,chr17-31336865-C-T,chr17-31336868-T-A,31336865,0.02,-38,-0.09,49,49,230,49,...,87,0,0.01,1.0,87,no_splicing,no_splicing,True,False,ss_lib
258,chr17-31336867-T-G,chr17-31336871-T-C,31336867,0.36,-5,-0.15,47,47,232,47,...,52,0,0.02,2.0,52,low_gain,low_gain,False,True,ss_lib


In [215]:
# some statistics
print(f"Max score dist {final_ss_lib["pangolin_gain_score_dist"].max()}")
print(f"Num variants which are not in the library at the same position as ss vars {sum(3-final_ss_lib["start"].value_counts())}")
print(f"Unique genomic positions positions {len(final_ss_lib["start"].unique())}")
print(f"Number of exons {final_ss_lib["exon_number"].value_counts()}")

Max score dist 0.21000000000000002
Num variants which are not in the library at the same position as ss vars -32
Unique genomic positions positions 76
Number of exons exon_number
17.0    82
25.0    46
42.0    46
34.0    38
39.0    34
27.0    14
Name: count, dtype: int64


ss variant library with bad marker variant

In [ ]:
# score only the marker variants with pangolin
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_ss_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, library)

syn_var_path = basepath + "/search_sequences/library_design/design_sequences/data/syn_vars_ss.vcf"
syn_var_file = open(syn_var_path, "w")
header_lines = [
    '##fileformat=VCFv4.1',
    '##fileDate=2025-08-31',
    '##reference=GRCh38',
    '##contig=<ID=chr1>',
    '##contig=<ID=chr2>',
    '##contig=<ID=chr3>',
    '##contig=<ID=chr4>',
    '##contig=<ID=chr5>',
    '##contig=<ID=chr6>',
    '##contig=<ID=chr7>',
    '##contig=<ID=chr8>',
    '##contig=<ID=chr9>',
    '##contig=<ID=chr10>',
    '##contig=<ID=chr11>',
    '##contig=<ID=chr12>',
    '##contig=<ID=chr13>',
    '##contig=<ID=chr14>',
    '##contig=<ID=chr15>',
    '##contig=<ID=chr16>',
    '##contig=<ID=chr17>',
    '##contig=<ID=chr18>',
    '##contig=<ID=chr19>',
    '##contig=<ID=chr20>',
    '##contig=<ID=chr21>',
    '##contig=<ID=chr22>',
    '##contig=<ID=chrX>',
    '##contig=<ID=chrY>',
    '##ID=<Description="Combined splice variant plus synonymous marker variant">',
    '#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO'
]
for line in header_lines:
    syn_var_file.write(line + "\n")

for syn_var in list(combined_variants_data["syn_var"].unique()):
    syn_var_parts = syn_var.split('-')
    chrom = syn_var_parts[0]
    pos = syn_var_parts[1]
    ref = syn_var_parts[2]
    alt = syn_var_parts[3]
    syn_var_file.write('\t'.join([chrom, pos, '.', ref, alt, '.', '.', '.']) + "\n")

syn_var_file.close()

/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)


In [ ]:
syn_vars_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/syn_vars_ss_pangolin.vcf"

syn_vars_data = []
with open(syn_vars_pangolin_path, "r") as syn_vars_pangolin_file:
    for line in syn_vars_pangolin_file:
        if line.startswith('#') or line.strip() == '':
            continue

        parts = line.split('\t')
        ##CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
        chrom = parts[0]
        pos = int(parts[1])
        ref = parts[3]
        alt = parts[4]
        info = parts[7]

        pangolin = functions.find_between(info, "Pangolin=", "(;|$)") # ENSG00000196712.18|-5:0.8100000023841858|25:-0.6399999856948853|Warnings:
        pangolin_parts = pangolin.split('|')
        pangolin_gain = pangolin_parts[1].split(':')
        pangolin_loss = pangolin_parts[2].split(':')

        ex_int = "intronic"
        distance_to_ss = 0
        distance_to_ss_left = 0
        distance_to_ss_right = 0
        for current_exon_number, exon_boundary in enumerate(exon_boundaries_oi):
            if exon_boundary[0] <= pos <= exon_boundary[1]:
                ex_int = "exonic"
                distance_to_ss_left = pos - exon_boundary[0]
                distance_to_ss_right = exon_boundary[1] - pos
                distance_to_ss = min(distance_to_ss_left, distance_to_ss_right)

        new_data = {
            "syn_var": '-'.join([chrom, str(pos), ref, alt]),
            "pangolin_gain_syn": round(float(pangolin_gain[1]), 2),
            "pangolin_gain_pos_syn": int(pangolin_gain[0]),
            "pangolin_loss_syn": round(float(pangolin_loss[1]), 2),
            "pangolin_loss_pos_syn": int(pangolin_loss[0]),
            "distance_to_ss_syn": distance_to_ss,
            "distance_to_ss_left_syn": distance_to_ss_left,
            "distance_to_ss_right_syn": distance_to_ss_right,
        }
        syn_vars_data.append(new_data)
syn_vars_data = pd.DataFrame(syn_vars_data)
syn_vars_data

,syn_var,pangolin_gain_syn,pangolin_gain_pos_syn,pangolin_loss_syn,pangolin_loss_pos_syn,distance_to_ss_syn,distance_to_ss_left_syn,distance_to_ss_right_syn
0,chr17-31225109-T-C,0.02,-14,-0.02,32,14,14,141
1,chr17-31225106-A-G,0.01,35,-0.08,144,11,11,144
2,chr17-31225097-G-A,0.09,2,-0.60,1,2,2,153
3,chr17-31225103-T-C,0.07,-8,-0.05,38,8,8,147
4,chr17-31225104-A-C,0.17,-9,-0.09,37,9,9,146
...,...,...,...,...,...,...,...,...
386,chr17-31336877-T-A,0.01,-242,-0.02,37,37,242,37
387,chr17-31336877-T-C,0.17,37,-0.03,-50,37,242,37
388,chr17-31336865-C-T,0.01,-38,-0.05,49,49,230,49
389,chr17-31336865-C-G,0.05,49,-0.01,49,49,230,49


In [ ]:
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_ss_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, library)

# filter for ss variants in specific exons 
test = library[library["exon_number"].isin([17,25,34,39,42,27])]
test = test[(test["pangolin_gain"] > 0.2) & (abs(test["pangolin_loss"]) < abs(test["pangolin_gain"])) & (test["splice_effect_to_canon_ss_dist"] > 10) | (test["start"] == 31233123)]
positions_oi = test["start"].unique()
print(len(positions_oi))

combined_variants_data = combined_variants_data[combined_variants_data["start"].isin(positions_oi)]
print(len(combined_variants_data["start"].unique()))


# require the splice variant to create a new ss which is at least 10 bp away from the closest canonical ss and allow no_splicing variants because they are from ClinVAr
# remove marker variants which push the predicted ss to the canonical ss. Assumes that the variant_oi itself does not influence the canonical ss.
preferred_synonymous_variants = combined_variants_data[(combined_variants_data["splice_effect_to_canon_ss_dist_combined"] >= 10) | (combined_variants_data["pangolin_gain_bin"] == "no_splicing")]

# remove variants with loss > gain
vars2remove = preferred_synonymous_variants[preferred_synonymous_variants["is_pangolin"]][~preferred_synonymous_variants[preferred_synonymous_variants["is_pangolin"]]["var_id"].isin(test["var_id"])]["var_id"]
preferred_synonymous_variants = preferred_synonymous_variants[~preferred_synonymous_variants["var_id"].isin(vars2remove)]

preferred_synonymous_variants = preferred_synonymous_variants.merge(syn_vars_data, on = "syn_var")
assert sum(preferred_synonymous_variants["pangolin_gain_syn"].isna()) == 0
preferred_synonymous_variants = preferred_synonymous_variants[preferred_synonymous_variants["pangolin_gain_syn"] < 0.2]



# order by combined_score and select the top x with the largest scores (strictly speaking this is more a distance than a score) -> these marker variants influence splicing the most
take_top_markers = 1
preferred_synonymous_variants = preferred_synonymous_variants.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_score_dist", ascending=False).iloc[0:take_top_markers]) # take top x
preferred_synonymous_variants = preferred_synonymous_variants.reset_index(drop=True)


# require that the pangolin gain score difference between combined variant and single variant oi is large -> bad marker
preferred_synonymous_variants = preferred_synonymous_variants[preferred_synonymous_variants["pangolin_gain_score_dist"] > 0.2]


final_ss_bad_marker_lib = preferred_synonymous_variants.copy()

# add ex27 specific bad marker variants that can be used later for comparison
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_2_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, library)
ex27ss_syn_vars = [
    "chr17-31233129-G-A",
    "chr17-31233159-A-G",
    "chr17-31233126-A-G",
    "chr17-31233075-C-A"
]
additional_variants = combined_variants_data[combined_variants_data["var_id"].isin(ex27ss_vars) & combined_variants_data["syn_var"].isin(ex27ss_syn_vars)]
final_ss_bad_marker_lib = pd.concat([final_ss_bad_marker_lib, additional_variants])
final_ss_bad_marker_lib = final_ss_bad_marker_lib.drop_duplicates()

final_ss_bad_marker_lib["source"] = "bad_marker_lib"
final_ss_bad_marker_lib


/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)


76
76


/tmp/ipykernel_3362068/1210535977.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  preferred_synonymous_variants = preferred_synonymous_variants.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_score_dist", ascending=False).iloc[0:take_top_markers]) # take top x
/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.group

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,is_clinvar,is_pangolin,pangolin_gain_syn,pangolin_gain_pos_syn,pangolin_loss_syn,pangolin_loss_pos_syn,distance_to_ss_syn,distance_to_ss_left_syn,distance_to_ss_right_syn,source
4,chr17-31225105-G-T,chr17-31225115-T-C,31225105,0.03,36,-0.09,145,10,10,145,...,False,True,0.11,-20.0,-0.13,26.0,20.0,20.0,135.0,bad_marker_lib
6,chr17-31225111-C-A,chr17-31225109-T-C,31225109,0.12,32,-0.05,-11,14,14,141,...,True,True,0.02,-14.0,-0.02,32.0,14.0,14.0,141.0,bad_marker_lib
7,chr17-31225112-C-A,chr17-31225121-T-C,31225112,0.10,29,-0.08,-14,17,17,138,...,False,True,0.09,-26.0,-0.14,20.0,26.0,26.0,129.0,bad_marker_lib
9,chr17-31225114-G-A,chr17-31225124-C-A,31225114,0.03,27,-0.07,136,19,19,136,...,True,True,0.07,-29.0,-0.10,17.0,29.0,29.0,126.0,bad_marker_lib
10,chr17-31225114-G-C,chr17-31225121-T-C,31225114,0.02,27,-0.04,-16,19,19,136,...,False,True,0.09,-26.0,-0.14,20.0,26.0,26.0,129.0,bad_marker_lib
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,chr17-31336865-C-A,chr17-31336862-G-C,31336862,0.02,-35,-0.09,52,52,227,52,...,False,True,0.15,52.0,-0.01,-35.0,52.0,227.0,52.0,bad_marker_lib
129,chr17-31336867-T-G,chr17-31336865-C-G,31336865,0.86,-3,-0.37,49,49,230,49,...,False,True,0.05,49.0,-0.01,49.0,49.0,230.0,49.0,bad_marker_lib
1111,chr17-31233072-A-G,chr17-31233075-C-A,31233072,0.38,1,-0.11,-70,70,70,141,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bad_marker_lib
1140,chr17-31233126-A-G,chr17-31233129-G-A,31233126,0.75,-1,-0.47,87,87,124,87,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bad_marker_lib


In [36]:
# chr17-31233126-A-G    R1207R  (A/G)TT(G/A)    syn_var: chr17-31233129-G-A
# chr17-31233156-T-G    D1217E  (T/G)CA(A/G)    syn_var: chr17-31233159-A-G
# chr17-31233123-G-A    E1206E  (G/A)AG(A/G)    syn_var: chr17-31233126-A-G
# chr17-31233072-A-G    Q1189Q  (A/G)GG(C/A)    syn_var: chr17-31233075-C-A

In [220]:
print(f"Max score dist {final_ss_bad_marker_lib["pangolin_gain_score_dist"].max()}")
print(f"Max combined pangolin gain {final_ss_bad_marker_lib["pangolin_gain_score_combined"].max()}")
print(f"Max pangolin gain {final_ss_bad_marker_lib["pangolin_gain"].max()}")
print(f"Number of unique positions {len(final_ss_bad_marker_lib["start"].unique())}")
print(f"Number of exons {final_ss_bad_marker_lib["exon_number"].value_counts()}")

Max score dist 0.87
Max combined pangolin gain 0.88
Max pangolin gain 0.89
Number of unique positions 57
Number of exons exon_number
17.0    19
34.0    12
25.0    11
42.0    10
27.0     8
39.0     8
Name: count, dtype: int64


stop gained library

In [ ]:
# read data
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_stop_gained_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, library)

# select marker variant with the smallest pangolin score difference. If there is a tie select the one with the smallest splice position.
# This makes sense because these variants are predicted to not influence splicing. Thus, the position where the largest splice gain is is not relevant here (and often very far away from the variant) and only used to resolve ties.
take_top_markers = 1
preferred_synonymous_variants = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values(["pangolin_gain_score_dist", "pangolin_gain_pos_dist"], ascending=True).iloc[0:take_top_markers]) # take top x
preferred_synonymous_variants = preferred_synonymous_variants.reset_index(drop=True)

final_stop_gained_lib = preferred_synonymous_variants.copy()
final_stop_gained_lib["source"] = "stop_gained_lib"
final_stop_gained_lib

/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)
/tmp/ipykernel_3362068/986856365.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  preferred_synonymous_variants = combined_variants_data.groupby("var_id").apply(la

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,splice_effect_to_canon_ss_dist,pangolin_gain_pos_dist,pangolin_gain_score_dist,pangolin_gain_combined_score,splice_effect_to_canon_ss_dist_combined,pangolin_gain_combined_bin,pangolin_gain_bin,is_clinvar,is_pangolin,source
0,chr17-31225145-T-A,chr17-31225151-T-C,31225145,0.00,-1,-0.24,105,50,50,105,...,49,0,0.00,0.0,49,no_splicing,no_splicing,False,False,stop_gained_lib
1,chr17-31225173-C-T,chr17-31225181-C-T,31225173,0.01,-37,-0.41,77,77,78,77,...,41,0,0.00,0.0,41,no_splicing,no_splicing,True,False,stop_gained_lib
2,chr17-31225198-T-A,chr17-31225205-T-C,31225198,0.01,-57,-0.20,52,52,103,52,...,46,0,0.00,0.0,46,no_splicing,no_splicing,True,False,stop_gained_lib
3,chr17-31225230-G-T,chr17-31225229-G-A,31225229,0.01,-88,-0.06,21,21,134,21,...,46,0,0.00,0.0,46,no_splicing,no_splicing,False,False,stop_gained_lib
4,chr17-31225233-A-T,chr17-31225241-C-T,31225233,0.01,41,-0.27,17,17,138,17,...,24,0,0.00,0.0,24,no_splicing,no_splicing,True,False,stop_gained_lib
5,chr17-31232083-C-T,chr17-31232076-T-C,31232076,0.01,10,-0.12,-3,3,3,113,...,13,0,0.00,0.0,13,no_splicing,no_splicing,True,False,stop_gained_lib
6,chr17-31232108-C-G,chr17-31232112-T-C,31232108,0.01,41,-0.06,81,35,35,81,...,40,0,0.00,0.0,40,no_splicing,no_splicing,True,False,stop_gained_lib
7,chr17-31232170-A-T,chr17-31232166-A-G,31232166,0.02,-17,-0.15,23,23,93,23,...,40,0,0.00,0.0,40,no_splicing,no_splicing,False,False,stop_gained_lib
8,chr17-31232176-C-T,chr17-31232181-A-G,31232176,0.05,-27,-0.29,13,13,103,13,...,40,0,0.00,0.0,40,no_splicing,no_splicing,True,False,stop_gained_lib
9,chr17-31232180-T-G,chr17-31232184-T-C,31232180,0.04,9,-0.08,9,9,107,9,...,0,0,0.00,0.0,0,no_splicing,no_splicing,True,False,stop_gained_lib


In [222]:
print(f"Max score dist {final_stop_gained_lib["pangolin_gain_score_dist"].max()}")
print(f"Max combined pangolin gain {final_stop_gained_lib["pangolin_gain_score_combined"].max()}")
print(f"Max pangolin gain {final_stop_gained_lib["pangolin_gain"].max()}")
print(f"Number of unique positions {len(final_stop_gained_lib["start"].unique())}")
print(f"Number of exons {final_stop_gained_lib["exon_number"].value_counts()}")

Max score dist 0.01
Max combined pangolin gain 0.09
Max pangolin gain 0.09
Number of unique positions 30
Number of exons exon_number
17.0    5
25.0    5
27.0    5
34.0    5
39.0    5
42.0    5
Name: count, dtype: int64


syn muts at stop positions library

In [ ]:
combined_variants_pangolin_path = basepath + "/search_sequences/library_design/design_sequences/data/combined_variants_syn_muts_pangolin.vcf"
combined_variants_data = read_combined_variants_data(combined_variants_pangolin_path, syn_mut_lib)

# select marker variant with the smallest pangolin score difference. If there is a tie select the one with the smallest splice position.
# This makes sense because these variants are predicted to not influence splicing. Thus, the position where the largest splice gain is is not relevant here and only used to resolve ties.
take_top_markers = 1
preferred_synonymous_variants = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values(["pangolin_gain_score_dist", "pangolin_gain_pos_dist"], ascending=True).iloc[0:take_top_markers]).reset_index(drop=True) # take top x

# select one synonymous variant per stop codon variant
preferred_synonymous_variants = preferred_synonymous_variants.groupby("stop_var_pos").apply(lambda x: x.sort_values("pangolin_gain_score_dist").iloc[0]).reset_index(drop=True) # pangolin_gain_score_combined

final_stop_syn_gained_lib = preferred_synonymous_variants.copy()
final_stop_syn_gained_lib["source"] = "stop_syn_gained_lib"
final_stop_syn_gained_lib

/tmp/ipykernel_3362068/479000613.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_variants_data = combined_variants_data.groupby("var_id").apply(lambda x: x.sort_values("pangolin_gain_combined_score", ascending=True)).reset_index(drop=True)
/tmp/ipykernel_3362068/2231273030.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  preferred_synonymous_variants = combined_variants_data.groupby("var_id").apply(l

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,splice_effect_to_canon_ss_dist,pangolin_gain_pos_dist,pangolin_gain_score_dist,pangolin_gain_combined_score,splice_effect_to_canon_ss_dist_combined,pangolin_gain_combined_bin,pangolin_gain_bin,is_clinvar,is_pangolin,source
0,chr17-31225145-T-C,chr17-31225151-T-C,31225145,0.04,-4,-0.02,-50,50,50,105,...,46,0,0.00,0.0,46,no_splicing,no_splicing,False,False,stop_syn_gained_lib
1,chr17-31225175-A-G,chr17-31225166-T-C,31225166,0.03,84,-0.00,108,71,71,84,...,0,0,0.00,0.0,0,no_splicing,no_splicing,True,False,stop_syn_gained_lib
2,chr17-31225197-T-C,chr17-31225190-T-C,31225190,0.05,60,-0.00,-1,60,95,60,...,0,0,0.00,0.0,0,no_splicing,no_splicing,True,False,stop_syn_gained_lib
3,chr17-31225232-A-C,chr17-31225226-G-C,31225226,0.01,-128,-0.06,24,24,131,24,...,0,3,0.00,3.0,3,no_splicing,no_splicing,False,False,stop_syn_gained_lib
4,chr17-31225235-A-G,chr17-31225241-C-T,31225235,0.01,39,-0.10,15,15,140,15,...,24,0,0.00,0.0,24,no_splicing,no_splicing,True,False,stop_syn_gained_lib
5,chr17-31232085-G-A,chr17-31232088-A-C,31232085,0.01,-12,-0.01,104,12,12,104,...,0,0,0.00,0.0,0,no_splicing,no_splicing,False,False,stop_syn_gained_lib
6,chr17-31232109-A-C,chr17-31232100-A-T,31232100,0.00,89,-0.01,49,27,27,89,...,0,0,0.00,0.0,0,no_splicing,no_splicing,True,False,stop_syn_gained_lib
7,chr17-31232172-A-G,chr17-31232166-A-G,31232166,0.03,23,-0.07,23,23,93,23,...,0,0,0.01,1.0,0,no_splicing,no_splicing,True,False,stop_syn_gained_lib
8,chr17-31232178-G-A,chr17-31232181-A-G,31232178,0.02,-29,-0.17,11,11,105,11,...,40,0,0.00,0.0,40,no_splicing,no_splicing,True,False,stop_syn_gained_lib
9,chr17-31232181-A-G,chr17-31232172-A-G,31232172,0.01,286,-0.09,5,17,99,17,...,269,0,0.00,0.0,269,no_splicing,no_splicing,False,False,stop_syn_gained_lib


In [226]:
print(f"Max score dist {final_stop_syn_gained_lib["pangolin_gain_score_dist"].max()}")
print(f"Max combined pangolin gain {final_stop_syn_gained_lib["pangolin_gain_score_combined"].max()}")
print(f"Max pangolin gain {final_stop_syn_gained_lib["pangolin_gain"].max()}")
print(f"Number of unique positions {len(final_stop_syn_gained_lib["start"].unique())}")
print(f"Number of exons {final_stop_syn_gained_lib["exon_number"].value_counts()}")

Max score dist 0.010000000000000002
Max combined pangolin gain 0.07
Max pangolin gain 0.07
Number of unique positions 30
Number of exons exon_number
17.0    5
25.0    5
27.0    5
34.0    5
39.0    5
42.0    5
Name: count, dtype: int64


Join variants into the final library

In [ ]:
# collect all that previous information and save to file
final_library = final_ss_lib.copy()
final_library = pd.concat([final_library, final_ss_bad_marker_lib])
final_library = pd.concat([final_library, final_stop_gained_lib])
final_library = pd.concat([final_library, final_stop_syn_gained_lib])

final_library["combined_var_id"] = final_library["var_id"] + "_" + final_library["syn_var"]
final_library = final_library.reset_index(drop = True)
final_library = final_library.groupby("combined_var_id").apply(lambda x: x.sort_values("source", ascending = False).iloc[0]).reset_index(drop = True)

final_library.to_csv(basepath + "/search_sequences/library_design/design_sequences/data/final_library.tsv", sep = "\t", index = False)
final_library

/tmp/ipykernel_3362068/3193970863.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_library = final_library.groupby("combined_var_id").apply(lambda x: x.sort_values("source", ascending = False).iloc[0]).reset_index(drop = True)


,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,pangolin_loss_pos_syn,distance_to_ss_syn,distance_to_ss_left_syn,distance_to_ss_right_syn,stop_var_id,stop_var_chrom,stop_var_pos,stop_var_ref,stop_var_alt,combined_var_id
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.10,43,-0.18,152,3,3,152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31225098-G-A_chr17-31225106-A-G
1,chr17-31225098-G-A,chr17-31225109-T-C,31225098,0.07,43,-0.04,-3,3,3,152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31225098-G-A_chr17-31225109-T-C
2,chr17-31225098-G-T,chr17-31225106-A-G,31225098,0.18,43,-0.29,152,3,3,152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31225098-G-T_chr17-31225106-A-G
3,chr17-31225098-G-T,chr17-31225109-T-C,31225098,0.19,43,-0.17,0,3,3,152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31225098-G-T_chr17-31225109-T-C
4,chr17-31225100-A-T,chr17-31225106-A-G,31225100,0.19,41,-0.07,-5,5,5,150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31225100-A-T_chr17-31225106-A-G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381,chr17-31336865-C-T,chr17-31336856-T-C,31336856,0.01,-29,-0.05,58,58,221,58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31336865-C-T_chr17-31336856-T-C
382,chr17-31336865-C-T,chr17-31336868-T-A,31336865,0.02,-38,-0.09,49,49,230,49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31336865-C-T_chr17-31336868-T-A
383,chr17-31336867-T-G,chr17-31336860-C-T,31336860,0.33,2,-0.19,54,54,225,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chr17-31336867-T-G_chr17-31336860-C-T
384,chr17-31336867-T-G,chr17-31336865-C-G,31336865,0.86,-3,-0.37,49,49,230,49,...,49.0,49.0,230.0,49.0,NaN,NaN,NaN,NaN,NaN,chr17-31336867-T-G_chr17-31336865-C-G


DESIGN PEG RNAS

In [ ]:
prime_design_input = []
for i, row in final_library.iterrows():
    combined_var_id = row["combined_var_id"]

    mut_oi = row["var_id"]
    mut_oi = mut_oi.split('-')

    syn_mut = row["syn_var"]
    syn_mut = syn_mut.split('-')

    chrom, wt_seq_start, wt_seq, mut_seq = functions.get_mutated_sequence([mut_oi,syn_mut], genome, format = "primedesign", flanking_dist = 300)

    prime_design_input.append({
        "combined_var_id": combined_var_id,
        "mut_seq": mut_seq
    })
prime_design_input = pd.DataFrame(prime_design_input)

prime_design_input.to_csv(basepath + "/search_sequences/library_design/design_sequences/data/final_library_prime_design_input_intermediate.csv", sep = ",", header = True, index = False) # save to file
prime_design_input

,combined_var_id,mut_seq
0,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...
1,chr17-31225098-G-A_chr17-31225109-T-C,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...
2,chr17-31225098-G-T_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...
3,chr17-31225098-G-T_chr17-31225109-T-C,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...
4,chr17-31225100-A-T_chr17-31225106-A-G,ACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAGAC...
...,...,...
391,chr17-31260484-G-T_chr17-31260492-T-C,GAAATGGAAAGCCAACTTTCTCCTTGTCCTTTTTGCTTTGTCTAAT...
392,chr17-31260489-A-G_chr17-31260492-T-C,GGAAAGCCAACTTTCTCCTTGTCCTTTTTGCTTTGTCTAATGTCAA...
393,chr17-31260487-A-T_chr17-31260492-T-A,ATGGAAAGCCAACTTTCTCCTTGTCCTTTTTGCTTTGTCTAATGTC...
394,chr17-31336793-C-T_chr17-31336796-C-T,CTTCTGTACTATAGCATATCTGTTTTATCATCAGGAGGTTTTTTGT...


ANALYZE PEG RNAS

In [ ]:
# read primedesign output
prime_design_output = pd.read_csv(basepath + "/search_sequences/library_design/design_sequences/data/251107_08.19.45_PrimeDesign/20251107_08.19.46_PrimeDesign.csv")
prime_design_output

,Target_name,Target_sequence,pegRNA_number,gRNA_type,Spacer_sequence,Spacer_GC_content,PAM_sequence,Extension_sequence,Strand,Annotation,...,ngRNA-to-pegRNA_distance,PBS_length,PBS_GC_content,RTT_length,RTT_GC_content,First_extension_nucleotide,Spacer_sequence_order_TOP,Spacer_sequence_order_BOTTOM,pegRNA_extension_sequence_order_TOP,pegRNA_extension_sequence_order_BOTTOM
0,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...,1,pegRNA,TTTATTTATTTTTTTCTAGC,0.15,AGG,TcCTATCTGtCTGCTAGAAAAAAATA,+,PAM_disrupted,...,NaN,11.0,0.09,15.0,0.47,T,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgA
1,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...,1,pegRNA,TTTATTTATTTTTTTCTAGC,0.15,AGG,TcCTATCTGtCTGCTAGAAAAAAATAA,+,PAM_disrupted,...,NaN,12.0,0.08,15.0,0.47,T,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcTcCTATCTGtCTGCTAGAAAAAAATAA,aaaaTTATTTTTTTCTAGCAGaCAGATAGgA
2,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...,1,pegRNA,TTTATTTATTTTTTTCTAGC,0.15,AGG,CTcCTATCTGtCTGCTAGAAAAAAATA,+,PAM_disrupted,...,NaN,11.0,0.09,16.0,0.50,C,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcCTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgAG
3,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...,1,pegRNA,TTTATTTATTTTTTTCTAGC,0.15,AGG,CTcCTATCTGtCTGCTAGAAAAAAATAA,+,PAM_disrupted,...,NaN,12.0,0.08,16.0,0.50,C,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcCTcCTATCTGtCTGCTAGAAAAAAATAA,aaaaTTATTTTTTTCTAGCAGaCAGATAGgAG
4,chr17-31225098-G-A_chr17-31225106-A-G,ATACAGTAGATTTATATTGTTGATGTTTTACTCATGTTTATGTCAG...,1,pegRNA,TTTATTTATTTTTTTCTAGC,0.15,AGG,ACTcCTATCTGtCTGCTAGAAAAAAATA,+,PAM_disrupted,...,NaN,11.0,0.09,17.0,0.47,A,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcACTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgAGT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34564,chr17-31336793-C-A_chr17-31336796-C-T,CTTCTGTACTATAGCATATCTGTTTTATCATCAGGAGGTTTTTTGT...,1020,ngRNA,GTTACTTTCTTAGTAGCCAC,0.40,AGG,NaN,+,PE3,...,18.0,NaN,NaN,NaN,NaN,NaN,caccGTTACTTTCTTAGTAGCCAC,aaacGTGGCTACTAAGAAAGTAAC,NaN,NaN
34565,chr17-31336793-C-A_chr17-31336796-C-T,CTTCTGTACTATAGCATATCTGTTTTATCATCAGGAGGTTTTTTGT...,1020,ngRNA,TCCCTTAGAGCTTCCACACA,0.50,TGG,NaN,+,PE3,...,48.0,NaN,NaN,NaN,NaN,NaN,caccGTCCCTTAGAGCTTCCACACA,aaacTGTGTGGAAGCTCTAAGGGAC,NaN,NaN
34566,chr17-31336793-C-A_chr17-31336796-C-T,CTTCTGTACTATAGCATATCTGTTTTATCATCAGGAGGTTTTTTGT...,1020,ngRNA,TAGAGCTTCCACACATGGAC,0.50,TGG,NaN,+,PE3,...,53.0,NaN,NaN,NaN,NaN,NaN,caccGTAGAGCTTCCACACATGGAC,aaacGTCCATGTGTGGAAGCTCTAC,NaN,NaN
34567,chr17-31336793-C-A_chr17-31336796-C-T,CTTCTGTACTATAGCATATCTGTTTTATCATCAGGAGGTTTTTTGT...,1020,ngRNA,TGTTCACAGCTTCATTTTAG,0.35,TGG,NaN,+,PE3,...,105.0,NaN,NaN,NaN,NaN,NaN,caccGTGTTCACAGCTTCATTTTAG,aaacCTAAAATGAAGCTGTGAACAC,NaN,NaN


In [ ]:
final_library_designed = final_library[final_library["combined_var_id"].isin(prime_design_output["Target_name"].unique())]
print(len(final_library_designed))
final_library_designed = final_library_designed.merge(prime_design_output, left_on = "combined_var_id", right_on = "Target_name")
final_library_designed

356


,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,ngRNA-to-pegRNA_distance,PBS_length,PBS_GC_content,RTT_length,RTT_GC_content,First_extension_nucleotide,Spacer_sequence_order_TOP,Spacer_sequence_order_BOTTOM,pegRNA_extension_sequence_order_TOP,pegRNA_extension_sequence_order_BOTTOM
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.1,43,-0.18,152,3,3,152,...,NaN,11.0,0.09,15.0,0.47,T,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgA
1,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.1,43,-0.18,152,3,3,152,...,NaN,12.0,0.08,15.0,0.47,T,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcTcCTATCTGtCTGCTAGAAAAAAATAA,aaaaTTATTTTTTTCTAGCAGaCAGATAGgA
2,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.1,43,-0.18,152,3,3,152,...,NaN,11.0,0.09,16.0,0.50,C,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcCTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgAG
3,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.1,43,-0.18,152,3,3,152,...,NaN,12.0,0.08,16.0,0.50,C,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcCTcCTATCTGtCTGCTAGAAAAAAATAA,aaaaTTATTTTTTTCTAGCAGaCAGATAGgAG
4,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.1,43,-0.18,152,3,3,152,...,NaN,11.0,0.09,17.0,0.47,A,caccGTTTATTTATTTTTTTCTAGCgtttt,ctctaaaacGCTAGAAAAAAATAAATAAAC,gtgcACTcCTATCTGtCTGCTAGAAAAAAATA,aaaaTATTTTTTTCTAGCAGaCAGATAGgAGT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34828,chr17-31336793-C-A,chr17-31336796-C-T,31336793,0.01,26,-0.26,-158,121,158,121,...,18.0,NaN,NaN,NaN,NaN,NaN,caccGTTACTTTCTTAGTAGCCAC,aaacGTGGCTACTAAGAAAGTAAC,NaN,NaN
34829,chr17-31336793-C-A,chr17-31336796-C-T,31336793,0.01,26,-0.26,-158,121,158,121,...,48.0,NaN,NaN,NaN,NaN,NaN,caccGTCCCTTAGAGCTTCCACACA,aaacTGTGTGGAAGCTCTAAGGGAC,NaN,NaN
34830,chr17-31336793-C-A,chr17-31336796-C-T,31336793,0.01,26,-0.26,-158,121,158,121,...,53.0,NaN,NaN,NaN,NaN,NaN,caccGTAGAGCTTCCACACATGGAC,aaacGTCCATGTGTGGAAGCTCTAC,NaN,NaN
34831,chr17-31336793-C-A,chr17-31336796-C-T,31336793,0.01,26,-0.26,-158,121,158,121,...,105.0,NaN,NaN,NaN,NaN,NaN,caccGTGTTCACAGCTTCATTTTAG,aaacCTAAAATGAAGCTGTGAACAC,NaN,NaN


In [ ]:
# filter the guide rnas

# remove nicking guides
final_library_designed = final_library_designed[~final_library_designed["Annotation"].str.contains("PE3")]

# remove pegRNAs which are too long for synthesis
#final_library_designed["pegRNA_length"] = 76+18+16 + final_library_designed["PBS_length"] + final_library_designed["RTT_length"] + [len(x) for x in final_library_designed["Spacer_sequence_order_TOP"]] + [len(x) for x in final_library_designed["Spacer_sequence_order_BOTTOM"]] + [len(x) for x in final_library_designed["pegRNA_extension_sequence_order_TOP"]] + [len(x) for x in final_library_designed["pegRNA_extension_sequence_order_BOTTOM"]]
#final_library_designed = final_library_designed[final_library_designed["pegRNA_length"] <= 300]

# add RHA length and filter peg rnas with too short RHA
final_library_designed["RHA_length"] = final_library_designed["Extension_sequence"].apply(lambda x: len(re.split("a|g|c|t", x)[2]))
final_library_designed = final_library_designed[final_library_designed["RHA_length"] > 5]

# remove poly T spacer and extension sequences
def find_longest_stretch(s, letter):
    return len(max(re.findall(letter + '+', s, flags=re.IGNORECASE), key = len))
final_library_designed["longest_T_stretch_spacer"] = final_library_designed["Spacer_sequence"].apply(lambda x: find_longest_stretch(x, "T"))
final_library_designed["longest_T_stretch_extension"] = final_library_designed["Extension_sequence"].apply(lambda x: find_longest_stretch(x, "T"))
final_library_designed = final_library_designed[final_library_designed["longest_T_stretch_spacer"] < 5]
final_library_designed = final_library_designed[final_library_designed["longest_T_stretch_extension"] < 5]

# these sequences should also not emerge when adding the constant sequences which are adjacent to the spacer / extension  in the final peg rna -- not checked here
final_library_designed = final_library_designed[~final_library_designed["Spacer_sequence"].str.contains("CGTCTC", case = False)]
final_library_designed = final_library_designed[~final_library_designed["Spacer_sequence"].str.contains("GAGACG", case = False)]
final_library_designed = final_library_designed[~final_library_designed["Extension_sequence"].str.contains("CGTCTC", case = False)]
final_library_designed = final_library_designed[~final_library_designed["Extension_sequence"].str.contains("GAGACG", case = False)]

# ensure RTT > 12: PBS == 12, RTT <= 12: PBS == 11 (based on 37119812) but keep all if no grnas are matching these expectations
final_library_designed_precursor = final_library_designed[((final_library_designed["PBS_length"] == 11) & (final_library_designed["RTT_length"] <= 12)) | ((final_library_designed["PBS_length"] == 12) & (final_library_designed["RTT_length"] > 12))]
final_library_missing = final_library_designed[~final_library_designed["combined_var_id"].isin(final_library_designed_precursor["combined_var_id"])]
final_library_designed = pd.concat([final_library_missing, final_library_designed_precursor])

# select the best one
final_library_designed["longest_T_stretch"] = final_library_designed[["longest_T_stretch_extension", "longest_T_stretch_spacer"]].max(axis = 1)
final_library_designed = final_library_designed.groupby("combined_var_id").apply(lambda x: x.sort_values(["Annotation", "RTT_length", "PBS_length", "RHA_length", "longest_T_stretch"]).iloc[0]).reset_index(drop = True)

final_library_designed

/tmp/ipykernel_3362068/1352460273.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_library_designed = final_library_designed.groupby("combined_var_id").apply(lambda x: x.sort_values(["Annotation", "RTT_length", "PBS_length", "RHA_length", "longest_T_stretch"]).iloc[0]).reset_index(drop = True)


,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,RTT_GC_content,First_extension_nucleotide,Spacer_sequence_order_TOP,Spacer_sequence_order_BOTTOM,pegRNA_extension_sequence_order_TOP,pegRNA_extension_sequence_order_BOTTOM,RHA_length,longest_T_stretch_spacer,longest_T_stretch_extension,longest_T_stretch
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098,0.10,43,-0.18,152,3,3,152,...,0.50,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGtC,22,1,3,3
1,chr17-31225098-G-A,chr17-31225109-T-C,31225098,0.07,43,-0.04,-3,3,3,152,...,0.50,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGtC,19,1,3,3
2,chr17-31225098-G-T,chr17-31225106-A-G,31225098,0.18,43,-0.29,152,3,3,152,...,0.50,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGaC,22,1,3,3
3,chr17-31225098-G-T,chr17-31225109-T-C,31225098,0.19,43,-0.17,0,3,3,152,...,0.50,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGaC,19,1,3,3
4,chr17-31225100-A-T,chr17-31225106-A-G,31225100,0.19,41,-0.07,-5,5,5,150,...,0.50,C,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcCtGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCaG,22,1,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,chr17-31336865-C-T,chr17-31336856-T-C,31336856,0.01,-29,-0.05,58,58,221,58,...,0.50,T,caccGTCCCTTAGAGCTTCCACACAgtttt,ctctaaaacTGTGTGGAAGCTCTAAGGGAC,gtgcTaACCAGTCCgTGTGTGGAAGCTCTA,aaaaTAGAGCTTCCACACAcGGACTGGTtA,15,2,1,2
332,chr17-31336865-C-T,chr17-31336868-T-A,31336865,0.02,-38,-0.09,49,49,230,49,...,0.33,T,caccGTAGAGCTTCCACACATGGACgtttt,ctctaaaacGTCCATGTGTGGAAGCTCTAC,gtgcTtATaACCAGTCCATGTGTGGAA,aaaaTTCCACACATGGACTGGTtATaA,18,2,2,2
333,chr17-31336867-T-G,chr17-31336860-C-T,31336860,0.33,2,-0.19,54,54,225,54,...,0.42,T,caccGTAGAGCTTCCACACATGGACgtttt,ctctaaaacGTCCATGTGTGGAAGCTCTAC,gtgcTAcTGACCAaTCCATGTGTGGAA,aaaaTTCCACACATGGAtTGGTCAgTA,13,2,1,2
334,chr17-31336867-T-G,chr17-31336865-C-G,31336865,0.86,-3,-0.37,49,49,230,49,...,0.50,T,caccGTAGAGCTTCCACACATGGACgtttt,ctctaaaacGTCCATGTGTGGAAGCTCTAC,gtgcTAcTcACCAGTCCATGTGTGGAA,aaaaTTCCACACATGGACTGGTgAgTA,18,2,1,2


In [242]:
# remove stop vars which do not have a matching synonymous variant
all_stop_vars = set(final_library_designed.loc[final_library_designed["source"] == "stop_gained_lib", "var_id"].unique())
all_stop_vars_with_syn = set(final_library_designed.loc[final_library_designed["source"] == "stop_syn_gained_lib", "stop_var_id"].unique())
print(len(all_stop_vars))
print(len(all_stop_vars_with_syn))
print(len(set(all_stop_vars) & set(all_stop_vars_with_syn)))
stop_vars_without_syn_var = all_stop_vars - all_stop_vars_with_syn
syn_vars_without_stop_var = all_stop_vars_with_syn - all_stop_vars 

final_library_designed = final_library_designed[~final_library_designed["var_id"].isin(stop_vars_without_syn_var)]
final_library_designed = final_library_designed[~final_library_designed["stop_var_id"].isin(syn_vars_without_stop_var)]


32
32
30


In [249]:
# add peg rnas from ex27 screen
final_library_designed["complete_peg_rna"] = ""
new_lines = [
    {"var_id": "chr17-31233016-A-T", "source": "ex27_lib_stop", "complete_peg_rna": "GTTGATCCGTCTCAACCGATTAGGCTTAGGTTACCACAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTTGTCTGGAGgTCCTaGTGGTAACCTAAGCCTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCTAGGACCTCCAGACAAGAGCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233128-T-A", "source": "ex27_lib_stop", "complete_peg_rna": "GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTGACCAGTTCaACCtATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233005-T-G", "source": "ex27_lib_stop", "complete_peg_rna": "GTTGATCCGTCTCAACCGTGTTTGTCCACATTAGGCTTGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCCTTGTGGTAgCCTcAGCCTAATGTGGACAATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCATTAGGCTGAGGCTACCACCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233157-C-T", "source": "ex27_lib_stop", "complete_peg_rna": "GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTAGGGAGTTCcCCTTaATCACCCATCATTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAAGGCACAGAATTTGCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233133-G-T", "source": "ex27_lib_stop", "complete_peg_rna": "GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCATTGTGACaAGTTaCACCAATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC"}
]
final_library_designed = pd.concat([final_library_designed, pd.DataFrame(new_lines)])
#chr17-31233016-A-T 29560034
# GTTGATCCGTCTCAACCGATTAGGCTTAGGTTACCACAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTTGTCTGGAGgTCCTaGTGGTAACCTAAGCCTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCTAGGACCTCCAGACAAGAGCGGGAGAGACGGTTCC
#chr17-31233128-T-A 29560146
# GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTGACCAGTTCaACCtATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC
#chr17-31233005-T-G 29560023
# GTTGATCCGTCTCAACCGTGTTTGTCCACATTAGGCTTGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCCTTGTGGTAgCCTcAGCCTAATGTGGACAATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACCATTAGGCTGAGGCTACCACCGGGAGAGACGGTTCC
#chr17-31233157-C-T 29560175
# GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTAGGGAGTTCcCCTTaATCACCCATCATTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAAGGCACAGAATTTGCGGGAGAGACGGTTCC
#chr17-31233133-G-T 29560151
# GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCATTGTGACaAGTTaCACCAATCTCTCAAACCGATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC

new_lines = [
    {"var_id": "chr17-31233072-A-G", "source": "ex27_lib_ss", "complete_peg_rna": "GTTGATCCGTCTCAACCGCTGACAAAAATCCTTCAACAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCAAATTCTGTtCCcTGTTGAAGGATTTTTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAGGGAACAGAATTTGCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233123-G-A", "source": "ex27_lib_ss", "complete_peg_rna": "GTTGATCCGTCTCAACCGCAGAAACAGTATTGGCTGATGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCGTTCCACCAAcCTtTCAAACCGATCAGCCAATACTGTTTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACGACAAGAGCTACATTTATGGCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233156-T-G", "source": "ex27_lib_ss", "complete_peg_rna": "GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCGGAGTTCTCCcTGcTCACCCATCATTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAAGGCACAGAATTTGCGGGAGAGACGGTTCC"},
    {"var_id": "chr17-31233126-A-G", "source": "ex27_lib_ss", "complete_peg_rna": "GTTGATCCGTCTCAACCGGGCTGATCGGTTTGAGAGATGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCACCAGTTCCACtAAcCTCTCAAACCGATCATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC"},
]
final_library_designed = pd.concat([final_library_designed, pd.DataFrame(new_lines)])

# chr17-31233072-A-G    29560090
# GTTGATCCGTCTCAACCGCTGACAAAAATCCTTCAACAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCTCAAATTCTGTtCCcTGTTGAAGGATTTTTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAGGGAACAGAATTTGCGGGAGAGACGGTTCC
# chr17-31233123-G-A    29560141
# GTTGATCCGTCTCAACCGCAGAAACAGTATTGGCTGATGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCGTTCCACCAAcCTtTCAAACCGATCAGCCAATACTGTTTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACGACAAGAGCTACATTTATGGCGGGAGAGACGGTTCC
# chr17-31233156-T-G    29560174
# GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCGGAGTTCTCCcTGcTCACCCATCATTGTTTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTCAACAAGGCACAGAATTTGCGGGAGAGACGGTTCC
# chr17-31233126-A-G    29560144
# GTTGATCCGTCTCAACCGGGCTGATCGGTTTGAGAGATGTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGGACCGAGTCGGTCCACCAGTTCCACtAAcCTCTCAAACCGATCATTTTTTTTGAACAGACGCAGGTGAACCGACGGATGATCTCGTGCACCGGTTCAAAAAAAACACCTGCGATCAAACTGTTCTCAGTGGGTAAGTGACGGGAGAGACGGTTCC


In [252]:
final_library_designed

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,First_extension_nucleotide,Spacer_sequence_order_TOP,Spacer_sequence_order_BOTTOM,pegRNA_extension_sequence_order_TOP,pegRNA_extension_sequence_order_BOTTOM,RHA_length,longest_T_stretch_spacer,longest_T_stretch_extension,longest_T_stretch,complete_peg_rna
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098.0,0.10,43.0,-0.18,152.0,3.0,3.0,152.0,...,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGtC,22.0,1.0,3.0,3.0,
1,chr17-31225098-G-A,chr17-31225109-T-C,31225098.0,0.07,43.0,-0.04,-3.0,3.0,3.0,152.0,...,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGtC,19.0,1.0,3.0,3.0,
2,chr17-31225098-G-T,chr17-31225106-A-G,31225098.0,0.18,43.0,-0.29,152.0,3.0,3.0,152.0,...,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGaC,22.0,1.0,3.0,3.0,
3,chr17-31225098-G-T,chr17-31225109-T-C,31225098.0,0.19,43.0,-0.17,0.0,3.0,3.0,152.0,...,G,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGaC,19.0,1.0,3.0,3.0,
4,chr17-31225100-A-T,chr17-31225106-A-G,31225100.0,0.19,41.0,-0.07,-5.0,5.0,5.0,150.0,...,C,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcCtGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCaG,22.0,1.0,3.0,3.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,chr17-31233133-G-T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GTTGATCCGTCTCAACCGTGATCGGTTTGAGAGATTGGGTTTTAGA...
0,chr17-31233072-A-G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GTTGATCCGTCTCAACCGCTGACAAAAATCCTTCAACAGTTTTAGA...
1,chr17-31233123-G-A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GTTGATCCGTCTCAACCGCAGAAACAGTATTGGCTGATGTTTTAGA...
2,chr17-31233156-T-G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GTTGATCCGTCTCAACCGGTCACAATGATGGGTGATCAGTTTTAGA...


In [ ]:
final_library_designed.to_csv(basepath + "/search_sequences/library_design/design_sequences/data/final_library_designed.tsv", sep = "\t")